## 1

In [14]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # Use GPU 1 which has 40GB free
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Reduce fragmentation

# Then your existing imports and code...

import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    GPT2Config
)
import timm
import Levenshtein
from tqdm import tqdm

BASE_DATA_DIR = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_3.pth"

# =====================================================================
# 1. DATASET DEFINITIONS
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor, transform=None, max_target_length=25):
        self.data_dir = data_dir            
        self.transform = transform
        self.processor = processor          
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"⚠️ Error: Path {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                
                word_id = parts[0]
                status = parts[1]
                if status != "ok":  
                    continue
                
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                id_parts = word_id.split('-')
                folder_grandparent = id_parts[0]                       
                folder_parent = f"{id_parts[0]}-{id_parts[1]}"         
                filename = f"{word_id}.png"                            
                
                img_path = os.path.join(self.data_dir, "words", folder_grandparent, folder_parent, filename)
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Successfully registered {len(self.samples)} valid image-text samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break 
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        
        if attempts >= len(self.samples):
            raise RuntimeError("Dataset Error: All image items appear corrupted.")

        if self.transform:
            image = self.transform(image)
            
        labels = self.processor(
            text, 
            padding='max_length', 
            max_length=self.max_target_length, 
            truncation=True, 
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        labels = torch.where(labels == self.processor.pad_token_id, torch.tensor(-100), labels)
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path, processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((64, 256)),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=None)
    total      = len(full_dataset)
    train_size = int(0.9 * total)

    generator = torch.Generator().manual_seed(42)
    indices   = torch.randperm(total, generator=generator).tolist()
    train_indices = indices[:train_size]
    val_indices   = indices[train_size:]

    train_samples = [full_dataset.samples[i] for i in train_indices]
    val_samples   = [full_dataset.samples[i] for i in val_indices]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=25):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (256, 64), color=255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(labels == self.processor.pad_token_id, torch.tensor(-100), labels)
            return image, labels, text

    train_dataset = SplitDataset(train_samples, processor, train_transform)
    val_dataset   = SplitDataset(val_samples, processor, val_transform)
    
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    val_loader    = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader


# =====================================================================
# 2. COMPLETE PIPELINE ARCHITECTURE 
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        # Safe structural initialization for regression weights
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        batch_size = x.size(0)
        points = self.fc(self.conv(x).view(batch_size, -1))
        return points.view(batch_size, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        control_points = self.localization(x)  
        B = x.size(0)
        
        cx = control_points[:, :, 0].mean(dim=1)  
        cy = control_points[:, :, 1].mean(dim=1)  
        
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05  
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]
        B, H, W, C = feat.shape
        feat = feat.reshape(B, H * W, C)
        feat = self.proj(feat)
        return feat.contiguous()


class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        config = GPT2Config.from_pretrained("gpt2")
        config.add_cross_attention = True
        self.decoder = GPT2LMHeadModel.from_pretrained(
            "gpt2",
            config=config
        )
        self.decoder.resize_token_embeddings(vocab_size)

    def forward(self, input_ids, encoder_hidden_states, labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn         = TPSSpatialTransformer(num_control_points)
        self.swin_encoder    = SwinEncoder(d_model=d_model)  
        self.bilstm          = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.gpt2_decoder = GPT2Decoder(vocab_size=vocab_size)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)
        pad_id = self.gpt2_decoder.decoder.config.eos_token_id  # 50256
        
        clean_ids = torch.where(target_ids == -100, torch.full_like(target_ids, pad_id), target_ids)
        
        bos_tokens = torch.full((clean_ids.size(0), 1), pad_id, dtype=torch.long, device=images.device)
        dec_input_ids = torch.cat([bos_tokens, clean_ids[:, :-1]], dim=1)
        
        outputs = self.gpt2_decoder(
            input_ids=dec_input_ids,
            encoder_hidden_states=memory,
            labels=None  
        )
        
        logits = outputs.logits
        
        if criterion is not None:
            return criterion(
                logits.reshape(-1, logits.size(-1)),
                target_ids.reshape(-1)
            )
        
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        return loss_fct(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))

    @torch.no_grad()
    def generate(self, images, max_length=25, bos_token_id=50256, eos_token_id=50256):
        device = images.device
        memory = self._extract_visual_memory(images)
        B = images.size(0)
        
        generated = torch.full(
            (B, 1),
            bos_token_id,
            dtype=torch.long,
            device=device
        )
        
        for _ in range(max_length - 1):
            outputs = self.gpt2_decoder(
                input_ids=generated,
                encoder_hidden_states=memory
            )
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            
            if (next_token.squeeze(-1) == eos_token_id).all():
                break
                
        return generated


# =====================================================================
# 3. POST-PROCESSING LOGIC AND ERROR METRICS
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = {"move", "to", "stop", "mr.", "gaitskell", "from"}
        if words_txt_file and os.path.exists(words_txt_file):
            self.build_lexicon_from_dataset(words_txt_file)
            
    def build_lexicon_from_dataset(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        self.lexicon.add(word)
        print(f"Agent state tracking verification module populated.")

    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        if (not cleaned or cleaned in self.lexicon or len(cleaned) <= 2 or any(c.isdigit() for c in cleaned)):
            return text_output
    
        best_candidate = text_output
        min_distance   = float('inf')
        target_len = len(cleaned)
        
        for legal_word in self.lexicon:
            if abs(len(legal_word) - target_len) > 1:
                continue   
            dist = Levenshtein.distance(cleaned, legal_word)
            if dist < min_distance and dist == 1:  
                min_distance   = dist
                best_candidate = legal_word
    
        if min_distance == 1:
            return best_candidate.upper() if text_output.isupper() else best_candidate
        return text_output


def calculate_metrics(predictions, targets):
    total_char_error_rate = 0.0
    incorrect_words = 0
    total_samples = len(targets)
    
    if total_samples == 0:
        return {"CER": 0.0, "WER": 0.0}

    for pred, target in zip(predictions, targets):
        pred_clean = pred.strip()
        target_clean = target.strip()
        
        edit_dist = Levenshtein.distance(pred_clean, target_clean)
        target_len = len(target_clean)
        
        if target_len > 0:
            total_char_error_rate += (edit_dist / target_len)
        else:
            if len(pred_clean) > 0:
                total_char_error_rate += 1.0 
        
        if edit_dist != 0:
            incorrect_words += 1

    cer = (total_char_error_rate / total_samples) * 100
    wer = (incorrect_words / total_samples) * 100
    return {"CER": cer, "WER": wer}


# =====================================================================
# 4. EARLY STOPPING CLASS
# =====================================================================
class EarlyStopping:
    """Stops training early if validation Word Error Rate (WER) doesn't improve."""
    def __init__(self, patience=4, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_wer = float('inf')
        self.early_stop = False

    def __call__(self, val_wer):
        if val_wer < self.best_wer - self.min_delta:
            self.best_wer = val_wer
            self.counter = 0  # reset tracking counter
            return True       # tells execution pipeline to save checkpoint
        else:
            self.counter += 1
            print(f" EarlyStopping Counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
            return False


# =====================================================================
# 5. TRAINING PIPELINE RUNNER
# =====================================================================
def run_training_pipeline(epochs=20, batch_size=32, lr=5e-5, patience=4):
    print("Loading GPT-2 tokenizer...")
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    
    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6)

    # Initialize early stopping tracker
    early_stopper = EarlyStopping(patience=patience, min_delta=0.01)

    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    swin_unfrozen       = False

    print(f"\nTraining on: {device}")
    for epoch in range(1, epochs + 1):

        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(), 'lr': 1e-6},
                {'params': model.swin_encoder.proj.parameters(), 'lr': 5e-5},
                {'params': model.bilstm.parameters(), 'lr': 5e-5},
                {'params': model.gpt2_decoder.parameters(), 'lr': 5e-5},
                {'params': model.tps_stn.parameters(), 'lr': 5e-5},
            ], weight_decay=0.05)
            
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6)
            swin_unfrozen = True
            print("Swin encoder unfrozen with conservative hyper-parameters.")

        # --- Train Phase ---
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            
            loss = model(images, target_ids=labels, criterion=criterion)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")

        # --- Validation Phase ---
        model.eval()
        all_predictions, all_targets = [], []
        first_batch_done = False  

        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens = model.generate(
                    images,
                    max_length=25,
                    bos_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
                                        
                if not first_batch_done:
                    print(f"\n--- DEBUG VISUALIZATION (Epoch {epoch} First Batch) ---")
                    for i in range(min(3, images.size(0))):
                        raw_debug = tokenizer.decode(tokens[i], skip_special_tokens=True)
                        print(f"Target: '{text_labels[i]}' | Pred: '{raw_debug.strip()}'")
                    print("----------------------------------------------------\n")
                    first_batch_done = True
                
                for i in range(images.size(0)):
                    raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                    verified = agent_verifier.verify_and_correct(raw)
                    all_predictions.append(verified)
                    all_targets.append(text_labels[i])
    
        metrics = calculate_metrics(all_predictions, all_targets)
        current_wer = metrics['WER']
        scheduler.step(current_wer) 
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        # Evaluate Early Stopping and Checkpoint updates
        is_best = early_stopper(current_wer)
        if is_best:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': early_stopper.best_wer,
                'cer': metrics['CER']
            }, CHECKPOINT_PATH)
            print(f"💾 Checkpoint saved | New Best WER: {early_stopper.best_wer:.2f}%")
        
        if early_stopper.early_stop:
            print(f"\n🛑 Early stopping triggered! Training stopped after epoch {epoch}.")
            break
            
        print("=" * 70)


if __name__ == "__main__":
    run_training_pipeline(epochs=50, batch_size=32, lr=5e-5, patience=5)

Loading GPT-2 tokenizer...
Successfully registered 38305 valid image-text samples.


Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['h.0.crossattention.c_attn.bias', 'h.0.crossattention.c_attn.weight', 'h.0.crossattention.c_proj.bias', 'h.0.crossattention.c_proj.weight', 'h.0.crossattention.q_attn.bias', 'h.0.crossattention.q_attn.weight', 'h.0.ln_cross_attn.bias', 'h.0.ln_cross_attn.weight', 'h.1.crossattention.c_attn.bias', 'h.1.crossattention.c_attn.weight', 'h.1.crossattention.c_proj.bias', 'h.1.crossattention.c_proj.weight', 'h.1.crossattention.q_attn.bias', 'h.1.crossattention.q_attn.weight', 'h.1.ln_cross_attn.bias', 'h.1.ln_cross_attn.weight', 'h.10.crossattention.c_attn.bias', 'h.10.crossattention.c_attn.weight', 'h.10.crossattention.c_proj.bias', 'h.10.crossattention.c_proj.weight', 'h.10.crossattention.q_attn.bias', 'h.10.crossattention.q_attn.weight', 'h.10.ln_cross_attn.bias', 'h.10.ln_cross_attn.weight', 'h.11.crossattention.c_attn.bias', 'h.11.crossattention.c_attn.weight', 'h.11.crossat

Agent state tracking verification module populated.

Training on: cuda


Epoch 1/50 [Train]: 100%|████████████████████████████████| 1077/1077 [03:09<00:00,  5.69it/s, loss=5.8659]


Epoch 1 | Train Loss: 5.7380


...

Epoch 6 | Train Loss: 3.1950


Epoch 6/50 [Val]:   1%|▍                                                  | 1/120 [00:00<01:19,  1.50it/s]


--- DEBUG VISUALIZATION (Epoch 6 First Batch) ---
Target: 'any' | Pred: 'any-any-any-any-any-any-any-any-any-any-any-any-'
Target: 'any' | Pred: 'any-any-any-any-any-any-any-any-any-any-any-any-'
Target: 'unopposed' | Pred: 'proposalsqueer-worded-worded-worded-worded-worded-worded-'
----------------------------------------------------



Epoch 6/50 [Val]: 100%|█████████████████████████████████████████████████| 120/120 [00:53<00:00,  2.26it/s]

Epoch 6 | CER: 1327.35% | WER: 100.00%
 EarlyStopping Counter: 5 out of 5

🛑 Early stopping triggered! Training stopped after epoch 6.


## 2

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Use GPU 1 which has 40GB free
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Reduce fragmentation

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForCausalLM
import timm
import Levenshtein
from tqdm import tqdm

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_gpt_2.pth"

# =====================================================================
# 2. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 transform=None, max_target_length=25):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                status        = parts[1]
                if status != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts           = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent      = f"{id_parts[0]}-{id_parts[1]}"
                filename           = f"{word_id}.png"
                img_path = os.path.join(
                    self.data_dir, "words",
                    folder_grandparent, folder_parent, filename
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images appear corrupted.")
        if self.transform:
            image = self.transform(image)
        
        # FIX: Append explicit EOS string so model learns when to stop generating
        text_with_eos = text + self.processor.eos_token
        
        labels = self.processor(
            text_with_eos,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        labels = torch.where(
            labels == self.processor.pad_token_id,
            torch.tensor(-100), labels
        )
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path, processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((64, 256)),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=None)
    total      = len(full_dataset)
    train_size = int(0.9 * total)

    generator     = torch.Generator().manual_seed(42)
    indices       = torch.randperm(total, generator=generator).tolist()
    train_indices = indices[:train_size]
    val_indices   = indices[train_size:]

    train_samples = [full_dataset.samples[i] for i in train_indices]
    val_samples   = [full_dataset.samples[i] for i in val_indices]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=25):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (256, 64), color=255)
            if self.transform:
                image = self.transform(image)
                
            text_with_eos = text + self.processor.eos_token
            labels = self.processor(
                text_with_eos,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_dataset = SplitDataset(train_samples, processor, train_transform)
    val_dataset   = SplitDataset(val_samples, processor, val_transform)
    
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    val_loader    = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE
# =====================================================================

class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B    = x.size(0)
        pts  = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:, :, 0].mean(dim=1)
        cy = control_points[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)  # (B, ~64, 512)
        return self.proj(feat)             # (B, ~64, 768)


class DistilRoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        print("Loading DistilRoBERTa pretrained weights...")
        distilroberta = RobertaModel.from_pretrained("distilroberta-base")
        config        = RobertaConfig.from_pretrained("distilroberta-base")

        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size

        self.decoder = RobertaForCausalLM(config)
        self.decoder.roberta.embeddings.load_state_dict(distilroberta.embeddings.state_dict())

        for i, layer in enumerate(self.decoder.roberta.encoder.layer):
            layer.load_state_dict(distilroberta.encoder.layer[i].state_dict(), strict=False)

        del distilroberta
        print("DistilRoBERTa: all 6 layers loaded successfully.")

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.decoder = DistilRoBERTaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)          # (B, 3, H, W)
        swin_feat = self.swin_encoder(rectified)  # (B, seq, 768)
        memory, _ = self.bilstm(swin_feat)         # (B, seq, 768)
        return memory

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        # FIX: Clean, standard causal sequence alignment shifting
        # Input to decoder: starts with BOS (0), drop final token to preserve original length dimension
        dec_input = target_ids.clone()
        dec_input = torch.where(dec_input == -100, torch.tensor(1, device=images.device), dec_input) # replace -100 with PAD(1)
        
        bos_tokens = torch.full((dec_input.size(0), 1), 0, dtype=torch.long, device=images.device) # BOS = 0
        dec_input_shifted = torch.cat([bos_tokens, dec_input[:, :-1]], dim=1)

        outputs = self.decoder(
            input_ids             = dec_input_shifted,
            encoder_hidden_states = memory,
            labels                = None   
        )
        logits = outputs.logits  # (B, T, vocab)

        if criterion is not None:
            loss = criterion(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
        else:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1), ignore_index=-100)
        return loss

    @torch.no_grad()
    def generate(self, images, max_length=25, bos_token_id=0, eos_token_id=2):
        # FIX: Replaced loop logic with standard, highly robust causal auto-regressive generation pass
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)
        
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        
        for _ in range(max_length - 1):
            outputs = self.decoder(input_ids=generated, encoder_hidden_states=memory)
            next_token_logits = outputs.logits[:, -1, :]
            next_tokens = next_token_logits.argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tokens], dim=1)
            
            # Stop early if all sequences in the batch have outputted the EOS token
            if (torch.any(generated == eos_token_id, dim=-1)).all():
                break
                
        return generated


# =====================================================================
# 4. AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = set()
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip()
                    if word:
                        self.lexicon.add(word.lower())
        print(f"Lexicon built: {len(self.lexicon)} words.")
    
    def verify_and_correct(self, text_output):
        cleaned    = text_output.strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
    
        best_candidate = text_output
        min_distance   = float('inf')
        target_len     = len(cleaned)
    
        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist < min_distance and dist == 1:
                min_distance   = dist
                best_candidate = word
    
        if min_distance == 1:
            if text_output.isupper():
                return best_candidate.upper()
            elif text_output[0].isupper():
                return best_candidate.capitalize()
            return best_candidate
        return text_output


# =====================================================================
# 5. METRICS & EARLY STOPPING
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        pred_c   = pred.strip()
        target_c = target.strip()
        dist     = Levenshtein.distance(pred_c, target_c)
        if len(target_c) > 0:
            total_cer += dist / len(target_c)
        elif len(pred_c) > 0:
            total_cer += 1.0
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_metric = float('inf')
        self.early_stop = False

    def __call__(self, current_metric):
        if current_metric < self.best_metric - self.min_delta:
            self.best_metric = current_metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 6. TRAINING
# =====================================================================
def run_training_pipeline(epochs=80, batch_size=32, lr=5e-5, early_stopping_patience=8, resume_from=None):
    print("Loading RoBERTa tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id} | PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_params   = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total_params/1e6:.2f}M")
    print(f"Trainable params: {trainable_params/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.05)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.7, patience=4, min_lr=5e-7, verbose=True
    )
    
    early_stopper = EarlyStopping(patience=8, min_delta=0.1)
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    best_val_wer   = float('inf')
    swin_unfrozen   = False

    print(f"\nTraining on: {device}")

    start_epoch = 1

    if resume_from and os.path.exists(resume_from):
        print(f"Resuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1

        if start_epoch > 4:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            swin_unfrozen = True
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(), 'lr': 5e-7}, 
                {'params': model.swin_encoder.proj.parameters(), 'lr': 1e-5},
                {'params': model.bilstm.parameters(), 'lr': 1e-5},
                {'params': model.decoder.parameters(), 'lr': 1e-5},
                {'params': model.tps_stn.parameters(), 'lr': 1e-5},
            ], weight_decay=0.05)
            print(f"Resumed at epoch {start_epoch} | Best WER so far: {best_val_wer:.2f}%")

    for epoch in range(start_epoch, epochs + 1):

        # Unfreeze Swin at epoch 4
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(), 'lr': 1e-6},
                {'params': model.swin_encoder.proj.parameters(), 'lr': lr},
                {'params': model.bilstm.parameters(), 'lr': lr},
                {'params': model.decoder.parameters(), 'lr': lr},
                {'params': model.tps_stn.parameters(), 'lr': lr},
            ], weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6, verbose=True
            )
            swin_unfrozen = True
            print("Swin encoder unfrozen.")

        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels, criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens = model.generate(
                    images,
                    max_length   = 25,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id
                )

                # Debug first batch each epoch
                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                        print(f"Target: '{text_labels[i]}' | Pred: '{raw.strip()}'")
                    print("----------------------------\n")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(tokens[i], skip_special_tokens=True)
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        scheduler.step(current_wer)
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer':             best_val_wer,
                'cer':                  metrics['CER']
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | New Best WER: {best_val_wer:.2f}%")
        
        if early_stopper(current_wer):
            print(f"\nEarly stopping triggered! Training stopped at epoch {epoch}.")
            break
            
        print("=" * 70)


if __name__ == "__main__":
    run_training_pipeline(epochs=80, batch_size=16, lr=5e-5, early_stopping_patience=8)

Loading RoBERTa tokenizer...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BOS=0 | EOS=2 | PAD=1
Registered 38305 samples.


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loading DistilRoBERTa pretrained weights...
DistilRoBERTa: all 6 layers loaded successfully.


...

Epoch 60 | Train Loss: 1.4174


Epoch 60/80 [Val]:   0%|▏                                                 | 1/240 [00:00<00:43,  5.46it/s]


--- Debug Epoch 60 ---
Target: 'any' | Pred: 'any'
Target: 'any' | Pred: 'any'
Target: 'unopposed' | Pred: 'proposed'
----------------------------



Epoch 60/80 [Val]: 100%|████████████████████████████████████████████████| 240/240 [00:18<00:00, 12.93it/s]

Epoch 60 | CER: 11.60% | WER: 19.89%
EarlyStopping counter: 8 out of 8

Early stopping triggered! Training stopped at epoch 60.


In [5]:
#test
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Use your target GPU

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForCausalLM
import timm
import Levenshtein
from tqdm import tqdm

# =====================================================================
# 1. CONFIGURATION & PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_gpt_2.pth"
BATCH_SIZE      = 32

# =====================================================================
# 2. DATASET LAYER
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor, max_target_length=25):
        self.data_dir          = data_dir
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                status        = parts[1]
                if status != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts           = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent      = f"{id_parts[0]}-{id_parts[1]}"
                filename           = f"{word_id}.png"
                img_path = os.path.join(
                    self.data_dir, "words",
                    folder_grandparent, folder_parent, filename
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)


class TestSplitDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (256, 64), color=255)
        if self.transform:
            image = self.transform(image)
        return image, text


def get_iam_test_dataloader(data_dir, words_txt_path, processor, batch_size=32):
    val_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor)
    total      = len(full_dataset)
    train_size = int(0.9 * total)

    # Replicates exact split logic from training (Seed 42)
    generator     = torch.Generator().manual_seed(42)
    indices       = torch.randperm(total, generator=generator).tolist()
    val_indices   = indices[train_size:]

    val_samples   = [full_dataset.samples[i] for i in val_indices]
    
    test_dataset = TestSplitDataset(val_samples, val_transform)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    return test_loader

# =====================================================================
# 3. ARCHITECTURE COMPONENTS
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B    = x.size(0)
        pts  = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:, :, 0].mean(dim=1)
        cy = control_points[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=False, # True/False doesn't matter since weights are loaded from your checkpoint anyway
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)
        return self.proj(feat)


class DistilRoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        config = RobertaConfig.from_pretrained("distilroberta-base")
        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size
        self.decoder               = RobertaForCausalLM(config)

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.decoder = DistilRoBERTaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory

    @torch.no_grad()
    def generate(self, images, max_length=25, bos_token_id=0, eos_token_id=2):
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        
        for _ in range(max_length - 1):
            outputs = self.decoder(input_ids=generated, encoder_hidden_states=memory)
            next_token_logits = outputs.logits[:, -1, :]
            next_tokens = next_token_logits.argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tokens], dim=1)
            if (torch.any(generated == eos_token_id, dim=-1)).all():
                break
        return generated

# =====================================================================
# 4. POST-PROCESSING MODULE & METRICS
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = set()
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip()
                    if word:
                        self.lexicon.add(word.lower())
        print(f"Lexicon built: {len(self.lexicon)} words.")
    
    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        if (not cleaned or cleaned in self.lexicon or len(cleaned) <= 2 or any(c.isdigit() for c in cleaned)):
            return text_output
    
        best_candidate = text_output
        min_distance   = float('inf')
        target_len     = len(cleaned)
    
        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist < min_distance and dist == 1:
                min_distance   = dist
                best_candidate = word
    
        if min_distance == 1:
            if text_output.isupper():
                return best_candidate.upper()
            elif text_output[0].isupper():
                return best_candidate.capitalize()
            return best_candidate
        return text_output


def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        pred_c   = pred.strip()
        target_c = target.strip()
        dist     = Levenshtein.distance(pred_c, target_c)
        if len(target_c) > 0:
            total_cer += dist / len(target_c)
        elif len(pred_c) > 0:
            total_cer += 1.0
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }

# =====================================================================
# 5. EXECUTION PIPELINE
# =====================================================================
def run_evaluation():
    print("Loading RoBERTa tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Initializing architecture model space...")
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"No valid checkpoint found at: {CHECKPOINT_PATH}")
        
    print(f"Loading weights from target model: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Successfully mounted weights. (Trained Epoch: {checkpoint.get('epoch', 'N/A')})")
    
    model.eval()
    
    test_loader = get_iam_test_dataloader(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=BATCH_SIZE)
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    all_raw_preds = []
    all_verified_preds = []
    all_targets = []
    
    print("\n--- Starting Model Inferencing Loop ---")
    with torch.no_grad():
        for images, text_labels in tqdm(test_loader, desc="Evaluating Steps"):
            images = images.to(device)
            
            tokens = model.generate(
                images,
                max_length=25,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
            for i in range(images.size(0)):
                raw_pred = tokenizer.decode(tokens[i], skip_special_tokens=True).strip()
                verified_pred = agent_verifier.verify_and_correct(raw_pred)
                
                all_raw_preds.append(raw_pred)
                all_verified_preds.append(verified_pred)
                all_targets.append(text_labels[i])

    raw_metrics = calculate_metrics(all_raw_preds, all_targets)
    verified_metrics = calculate_metrics(all_verified_preds, all_targets)
    
    print("\n" + "="*50)
    print("               FINAL EVALUATION SUMMARY")
    print("="*50)
    print(f"Total Test Samples Checked: {len(all_targets)}")
    print("-"*50)
    print(" Raw Outputs (Before Correction):")
    print(f"  -> Character Error Rate (CER): {raw_metrics['CER']:.2f}%")
    print(f"  -> Word Error Rate (WER)     : {raw_metrics['WER']:.2f}%")
    print("-"*50)
    print(" Verified Outputs (After Lexicon Fix):")
    print(f"  -> Character Error Rate (CER): {verified_metrics['CER']:.2f}%")
    print(f"  -> Word Error Rate (WER)     : {verified_metrics['WER']:.2f}%")
    print("="*50 + "\n")
    
    print("--- Sample Comparisons ---")
    sample_indices = random.sample(range(len(all_targets)), min(8, len(all_targets)))
    for idx in sample_indices:
        print(f"Ground Truth : '{all_targets[idx]}'")
        print(f"Model Raw    : '{all_raw_preds[idx]}'")
        print(f"Agent Fixed  : '{all_verified_preds[idx]}'")
        print("-" * 40)

if __name__ == "__main__":
    run_evaluation()

Loading RoBERTa tokenizer...
Initializing architecture model space...
Loading weights from target model: /home/mca24-26/handwritten text detection/iam_htr_gpt_2.pth


/tmp/ipykernel_3598749/1593550643.py:314: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)


Successfully mounted weights. (Trained Epoch: 52)
Registered 38305 samples.
Lexicon built: 6231 words.

--- Starting Model Inferencing Loop ---


Evaluating Steps: 100%|█████████████████████████████████████████████████| 120/120 [00:21<00:00,  5.47it/s]



               FINAL EVALUATION SUMMARY
Total Test Samples Checked: 3831
--------------------------------------------------
 Raw Outputs (Before Correction):
  -> Character Error Rate (CER): 11.50%
  -> Word Error Rate (WER)     : 19.92%
--------------------------------------------------
 Verified Outputs (After Lexicon Fix):
  -> Character Error Rate (CER): 11.58%
  -> Word Error Rate (WER)     : 19.66%

--- Sample Comparisons ---
Ground Truth : 'equipped'
Model Raw    : 'equipped'
Agent Fixed  : 'equipped'
----------------------------------------
Ground Truth : 'a'
Model Raw    : 'a'
Agent Fixed  : 'a'
----------------------------------------
Ground Truth : 'has'
Model Raw    : 'he'
Agent Fixed  : 'he'
----------------------------------------
Ground Truth : 'is'
Model Raw    : 'is'
Agent Fixed  : 'is'
----------------------------------------
Ground Truth : 'most'
Model Raw    : 'most'
Agent Fixed  : 'most'
----------------------------------------
Ground Truth : 'in'
Model Raw    : '

## 3

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, GPT2LMHeadModel,
                          GPT2Config)
import timm
import Levenshtein
from tqdm import tqdm
from collections import defaultdict

# =====================================================================
# 1. CONFIGURATION
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth"

# Increased resolution for better character discrimination
IMG_HEIGHT  = 96
IMG_WIDTH   = 384
MAX_SEQ_LEN = 25
BEAM_SIZE   = 5
D_MODEL     = 768
BATCH_SIZE  = 16   # reduced because higher resolution needs more VRAM

# =====================================================================
# 2. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=BATCH_SIZE):
    train_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.RandomApply([T.ColorJitter(
            brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(
        data_dir, words_txt_path, processor=processor
    )
    total      = len(full_dataset)
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(42)
    indices    = torch.randperm(total, generator=generator).tolist()

    train_samples = [full_dataset.samples[i]
                     for i in indices[:train_size]]
    val_samples   = [full_dataset.samples[i]
                     for i in indices[train_size:]]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor,
                     transform, max_target_length=MAX_SEQ_LEN):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB',
                                  (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_loader = DataLoader(
        SplitDataset(train_samples, processor, train_transform),
        batch_size=batch_size, shuffle=True,
        num_workers=4, drop_last=True
    )
    val_loader = DataLoader(
        SplitDataset(val_samples, processor, val_transform),
        batch_size=batch_size, shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B   = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        cp             = self.localization(x)
        cx             = cp[:, :, 0].mean(dim=1)
        cy             = cp[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False,
                             padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        # Stage 3: 512 channels, richer sequence at higher res
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))  # (B, seq, d_model)


# ── GPT-2 Decoder ────────────────────────────────────────────────────
# GPT-2 is natively causal — no bidirectional mismatch like RoBERTa
# Cross-attention layers added for visual conditioning
# ─────────────────────────────────────────────────────────────────────
class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        print("Loading GPT-2 pretrained weights...")

        PRETRAINED_VOCAB = 50257  # GPT-2 original vocab size

        # Step 1: Build target config with cross-attn + new vocab
        config                     = GPT2Config.from_pretrained("gpt2")
        config.add_cross_attention = True
        config.vocab_size          = vocab_size  # 50265

        # Step 2: Initialize model with new config (random weights)
        self.decoder = GPT2LMHeadModel(config)

        # Step 3: Load pretrained state dict
        pretrained      = GPT2LMHeadModel.from_pretrained("gpt2")
        pretrained_dict = pretrained.state_dict()

        # Step 4: Filter out keys that have size mismatches
        # These are the embedding and lm_head — handled manually below
        mismatch_keys = {
            "transformer.wte.weight",
            "lm_head.weight"
        }
        filtered_dict = {
            k: v for k, v in pretrained_dict.items()
            if k not in mismatch_keys
        }

        # Step 5: Load filtered weights — no size conflicts now
        # Missing keys = cross-attention layers (random init, correct)
        # Unexpected keys = none (we only removed mismatched ones)
        load_result = self.decoder.load_state_dict(
            filtered_dict, strict=False
        )
        print(f"  Loaded: {len(filtered_dict)} weight tensors")
        print(f"  Missing (new layers): {len(load_result.missing_keys)}")

        # Step 6: Copy pretrained embeddings for original 50257 tokens
        # New 8 tokens (50257-50264) stay randomly initialized
        with torch.no_grad():
            self.decoder.transformer.wte.weight\
                .data[:PRETRAINED_VOCAB].copy_(
                    pretrained_dict["transformer.wte.weight"]
                )
            self.decoder.lm_head.weight\
                .data[:PRETRAINED_VOCAB].copy_(
                    pretrained_dict["lm_head.weight"]
                )

        del pretrained, pretrained_dict
        print(f"GPT-2 weights loaded successfully.")
        print(f"  Vocab: {PRETRAINED_VOCAB} pretrained "
              f"+ {vocab_size - PRETRAINED_VOCAB} new tokens")
        print(f"  Cross-attention layers: randomly initialized")

    def forward(self, input_ids,
                encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL,
                 num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.gpt2_decoder = GPT2Decoder(
            vocab_size=vocab_size, d_model=d_model
        )

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()   

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        # Teacher forcing with correct manual shift
        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),
            dec_input
        )
        labels = target_ids[:, 1:].clone()

        outputs = self.gpt2_decoder(
            input_ids             = dec_input,
            encoder_hidden_states = memory,
            labels                = None
        )
        logits = outputs.logits

        if criterion is not None:
            return criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100
        )

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2,
                 beam_size=BEAM_SIZE):
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)
    
        # ── Fast greedy path (beam_size=1) ───────────────────
        # Processes entire batch in parallel — much faster
        if beam_size == 1:
            generated = torch.full(
                (B, 1), bos_token_id,
                dtype=torch.long, device=device
            )
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_length - 1):
                out         = self.gpt2_decoder(
                    input_ids             = generated,
                    encoder_hidden_states = memory,
                    labels                = None
                )
                next_tokens = out.logits[:, -1, :].argmax(
                    dim=-1, keepdim=True
                )
                next_tokens = next_tokens.masked_fill(
                    finished.unsqueeze(1), eos_token_id
                )
                generated   = torch.cat([generated, next_tokens], dim=1)
                finished   |= (next_tokens.squeeze(1) == eos_token_id)
                if finished.all():
                    break
            return generated[:, 1:]  # strip BOS
    
        # ── Beam search path (beam_size > 1) ─────────────────
        # Used only at final test time
        all_results = []
        for b in range(B):
            mem       = memory[b:b+1]
            beams     = [(0.0, [bos_token_id])]
            completed = []
    
            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor(
                        [tokens], dtype=torch.long, device=device
                    )
                    out      = self.gpt2_decoder(
                        input_ids             = inp,
                        encoder_hidden_states = mem,
                        labels                = None
                    )
                    log_prob = F.log_softmax(
                        out.logits[0, -1], dim=-1
                    )
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(),
                                       top_id.tolist()):
                        candidates.append(
                            (score + lp, tokens + [tid])
                        )
    
                if not candidates:
                    break
    
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break
    
                if not beams:
                    break
    
            all_beams        = completed if completed else beams
            _, best_tokens   = max(all_beams, key=lambda x: x[0])
            result           = torch.tensor(
                best_tokens[1:], dtype=torch.long, device=device
            )
            all_results.append(result)
    
        max_len = max(r.size(0) for r in all_results)
        padded  = torch.full(
            (B, max_len), eos_token_id,
            dtype=torch.long, device=device
        )
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded


# =====================================================================
# 4. STRONGER AGENTIC VERIFICATION MODULE
# Improvements over baseline:
# - Frequency-weighted candidate scoring
# - Confidence gating (skip correction if model is confident)
# - Context-aware: skips proper nouns (capitalized words)
# - Casing preservation
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon  = {}   # word -> frequency
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = defaultdict(int)
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] += 1
        self.lexicon  = dict(freq)
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon: {len(self.lexicon)} words | "
              f"Max freq: {self.freq_max}")

    def verify_and_correct(self, text_output,
                           top_logprob=None,
                           logprob_threshold=-0.3):
        
        cleaned = text_output.strip().lower()

        # Skip: empty, already valid, very short, has digits
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        # Skip proper nouns — capitalized words are likely correct
        if text_output[0].isupper() and len(text_output) > 1:
            if text_output[1:].islower():
                # Looks like a proper noun — trust the model
                if cleaned in self.lexicon:
                    return text_output

        # Confidence gate — if model was very confident, trust it
        if top_logprob is not None:
            avg_lp = sum(top_logprob) / max(len(top_logprob), 1)
            if avg_lp > logprob_threshold:
                return text_output

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue
            # Frequency-weighted score
            # Prefer common words when multiple edit-1 exist
            freq_score = freq / self.freq_max
            score      = freq_score - (dist * 0.3)
            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        # Preserve casing
        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# 5. METRICS & EARLY STOPPING
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 6. LLRD OPTIMIZER
# =====================================================================
def build_llrd_optimizer(model, base_lr=5e-5,
                         decay_factor=0.75,
                         weight_decay=0.05):
    
    assigned = set()

    def collect(named_params, filter_fn):
        params = []
        for name, param in named_params:
            if id(param) not in assigned and filter_fn(name):
                params.append(param)
                assigned.add(id(param))
        return params

    def collect_all(named_params):
        params = []
        for name, param in named_params:
            if id(param) not in assigned:
                params.append(param)
                assigned.add(id(param))
        return params

    # GPT-2 cross-attention layers (randomly initialized)
    gpt2_crossattn = collect(
        model.gpt2_decoder.named_parameters(),
        lambda n: "crossattention" in n or "cross_attn" in n
    )
    # GPT-2 self-attention + FFN (pretrained)
    gpt2_selfattn = collect_all(
        model.gpt2_decoder.named_parameters()
    )
    # BiLSTM
    bilstm_params = collect_all(model.bilstm.named_parameters())
    # Swin projection
    swin_proj = collect(
        model.swin_encoder.named_parameters(),
        lambda n: n.startswith("proj.") or n.startswith("norm.")
    )
    # Swin stages by depth
    swin_s4 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_3" in n
    )
    swin_s3 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_2" in n
    )
    swin_s2 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_1" in n
    )
    # Remaining Swin params (stage 1 + patch embed)
    swin_s1 = collect_all(
        model.swin_encoder.swin.named_parameters()
    )
    # TPS-STN
    tps_params = collect_all(model.tps_stn.named_parameters())

    lr = [
        base_lr,
        base_lr * decay_factor,
        base_lr * decay_factor**2,
        base_lr * decay_factor**3,
        base_lr * decay_factor**4,
        base_lr * decay_factor**5,
        base_lr * decay_factor**6,
        base_lr * decay_factor**7,
        base_lr * decay_factor**3,
    ]

    groups_raw = [
        (gpt2_crossattn, lr[0], "gpt2_crossattn"),
        (gpt2_selfattn,  lr[1], "gpt2_selfattn"),
        (bilstm_params,  lr[2], "bilstm"),
        (swin_proj,      lr[3], "swin_proj"),
        (swin_s4,        lr[4], "swin_stage4"),
        (swin_s3,        lr[5], "swin_stage3"),
        (swin_s2,        lr[6], "swin_stage2"),
        (swin_s1,        lr[7], "swin_stage1"),
        (tps_params,     lr[8], "tps_stn"),
    ]

    param_groups = [
        {"params": params, "lr": lrate, "name": name}
        for params, lrate, name in groups_raw
        if len(params) > 0
    ]

    # Verify complete coverage
    total_assigned = sum(
        len(g["params"]) for g in param_groups
    )
    total_model = sum(1 for _ in model.parameters())
    if total_assigned != total_model:
        print(f"WARNING: {total_model - total_assigned} "
              f"params unassigned")

    print("\nLLRD Groups:")
    print(f"{'Name':<20} {'LR':>10} {'Params':>10}")
    print("-" * 44)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<20} {g['lr']:>10.2e} "
              f"{n/1e6:>9.2f}M")

    return torch.optim.AdamW(
        param_groups, weight_decay=weight_decay
    )


# =====================================================================
# 7. TRAINING
# =====================================================================
def run_training_pipeline(epochs=60, batch_size=BATCH_SIZE,
                          base_lr=5e-5,
                          early_stopping_patience=8,
                          resume_from=None):

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | "
          f"EOS={tokenizer.eos_token_id} | "
          f"PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE,
        tokenizer, batch_size=batch_size
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model  = CompleteHTRPipeline(
        vocab_size=tokenizer.vocab_size
    ).to(device)

    total_p = sum(p.numel() for p in model.parameters())
    print(f"Total params: {total_p/1e6:.2f}M")
    print(f"Resolution  : {IMG_HEIGHT}x{IMG_WIDTH}")
    print(f"Beam size   : {BEAM_SIZE}")

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100, label_smoothing=0.1
    )

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    # Build LLRD optimizer over trainable params
    optimizer = build_llrd_optimizer(
        model, base_lr=base_lr,
        decay_factor=0.75, weight_decay=0.05
    )

    # CosineAnnealingWarmRestarts
    # T_0=10: restart every 10 epochs initially
    # T_mult=2: each restart doubles the cycle length
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )

    agent_verifier = AgenticVerificationModule(
        words_txt_file=WORDS_TXT_FILE
    )
    early_stopper  = EarlyStopping(
        patience=early_stopping_patience, min_delta=0.05
    )
    best_val_wer   = float('inf')
    start_epoch    = 1
    swin_unfrozen  = False

    # Resume from checkpoint
    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        if start_epoch > 4:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            swin_unfrozen = True
        print(f"Resumed epoch {start_epoch} | "
              f"Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")

    for epoch in range(start_epoch, epochs + 1):

        # Unfreeze Swin at epoch 4
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            # Rebuild optimizer now that all params are active
            optimizer = build_llrd_optimizer(
                model, base_lr=base_lr,
                decay_factor=0.75, weight_decay=0.05
            )
            scheduler = torch.optim.lr_scheduler\
                .CosineAnnealingWarmRestarts(
                    optimizer, T_0=10, T_mult=2, eta_min=1e-7
                )
            swin_unfrozen = True
            print("Swin encoder unfrozen — LLRD optimizer rebuilt.")

        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels,
                         criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        # Show decoder and swin LRs
        lrs = {g["name"]: g["lr"]
               for g in optimizer.param_groups}
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | "
              f"LR gpt2_cross: {lrs.get('gpt2_crossattn', 0):.2e} "
              f"| LR swin4: {lrs.get('swin_stage4', 0):.2e}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False
        
        with torch.no_grad():
            for images, _, text_labels in tqdm(
                    val_loader,
                    desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
        
                # Use greedy (beam_size=1) during training validation
                # Beam search only at final test time
                # This reduces val time from ~90min to ~2min per epoch
                tokens = model.generate(
                    images,
                    max_length   = MAX_SEQ_LEN,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id,
                    beam_size    = 1        # GREEDY during training
                )
                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(
                            tokens[i], skip_special_tokens=True
                        )
                        print(f"  Target: '{text_labels[i]}' "
                              f"| Pred: '{raw.strip()}'")
                    print("---")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(
                        tokens[i], skip_special_tokens=True
                    )
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | "
              f"CER: {metrics['CER']:.2f}% | "
              f"WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer':             best_val_wer,
                'cer':                  metrics['CER'],
                'img_height':           IMG_HEIGHT,
                'img_width':            IMG_WIDTH,
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | WER: {best_val_wer:.2f}%")

        if early_stopper(current_wer):
            print(f"Early stopping at epoch {epoch}.")
            break

        print("=" * 70)


# if __name__ == "__main__":
#     run_training_pipeline(
#         epochs                  = 60,
#         batch_size              = BATCH_SIZE,
#         base_lr                 = 5e-5,
#         early_stopping_patience = 8,
#         resume_from             = CHECKPOINT_PATH
#         # To resume: resume_from=CHECKPOINT_PATH
#     )

In [2]:
#test
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer
from PIL import Image
import Levenshtein
from tqdm import tqdm
import numpy as np
from collections import defaultdict

# =====================================================================
# TEST CONFIGURATION
# =====================================================================
TEST_CONFIG = {
    "checkpoint_path": "/home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth",
    "batch_size": 32,  # Can be larger for testing since no gradients
    "num_workers": 4,
    "beam_size": 5,  # Beam search for better accuracy
    "use_beam_search": True,  # Set to True for beam search, False for greedy
    "save_predictions": True,
    "output_dir": "./test_results_gpt",
    "test_ratio": 0.1,  # Use 10% of data for testing
}

# =====================================================================
# TEST DATASET (Replicates validation dataset but with test split)
# =====================================================================
class IAMTestDataset:
    def __init__(self, data_dir, words_txt_path, processor, transform=None, 
                 max_target_length=25, test_ratio=0.1):
        self.data_dir = data_dir
        self.processor = processor
        self.transform = transform
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_test_set(words_txt_path, test_ratio)
    
    def _parse_test_set(self, words_txt_path, test_ratio):
        if not os.path.exists(words_txt_path):
            print(f"Error: {words_txt_path} not found.")
            return
        
        all_samples = []
        with open(words_txt_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9 or parts[1] != "ok":
                    continue
                
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                word_id = parts[0]
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    all_samples.append((img_path, transcription))
        
        # Use last test_ratio as test set (consistent with training)
        test_size = int(len(all_samples) * test_ratio)
        self.samples = all_samples[-test_size:] if test_size > 0 else all_samples
        print(f"Test samples loaded: {len(self.samples)}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (384, 96), color=255)
        
        if self.transform:
            image = self.transform(image)
        
        return image, text, img_path


def get_test_dataloader(data_dir, words_txt_path, processor, 
                       batch_size=32, img_height=96, img_width=384):
    """Create test dataloader"""
    test_transform = T.Compose([
        T.Resize((img_height, img_width)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], 
                   std=[0.229, 0.224, 0.225])
    ])
    
    test_dataset = IAMTestDataset(
        data_dir, words_txt_path, processor, 
        transform=test_transform, 
        test_ratio=TEST_CONFIG["test_ratio"]
    )
    
    def collate_fn(batch):
        images = torch.stack([b[0] for b in batch])
        texts = [b[1] for b in batch]
        paths = [b[2] for b in batch]
        return images, texts, paths
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        collate_fn=collate_fn
    )
    return test_loader


# =====================================================================
# ENHANCED METRICS
# =====================================================================
def calculate_detailed_metrics(predictions, targets):
    """Calculate detailed metrics including character-level and word-level stats"""
    if not targets:
        return {}
    
    total_chars_correct = 0
    total_chars_target = 0
    total_chars_pred = 0
    total_words_correct = 0
    total_words = len(targets)
    
    cer_per_sample = []
    wer_per_sample = []
    word_distances = []
    
    for pred, target in zip(predictions, targets):
        pred = pred.strip().lower()
        target = target.strip().lower()
        
        # Character-level
        dist = Levenshtein.distance(pred, target)
        target_len = len(target)
        if target_len > 0:
            cer = dist / target_len
            cer_per_sample.append(cer)
            total_chars_correct += (target_len - dist)
            total_chars_target += target_len
            total_chars_pred += len(pred)
        
        # Word-level
        word_distances.append(dist)
        if pred == target:
            total_words_correct += 1
            wer_per_sample.append(0.0)
        else:
            wer_per_sample.append(1.0)
    
    avg_cer = np.mean(cer_per_sample) * 100 if cer_per_sample else 0
    avg_wer = np.mean(wer_per_sample) * 100 if wer_per_sample else 0
    cer_standard = (total_chars_target - total_chars_correct) / total_chars_target * 100 if total_chars_target > 0 else 0
    
    return {
        "CER": round(avg_cer, 4),
        "CER_standard": round(cer_standard, 4),
        "WER": round(avg_wer, 4),
        "Word_Accuracy": round((total_words_correct / total_words) * 100, 4),
        "Total_Samples": total_words,
        "Correct_Words": total_words_correct,
        "Avg_Target_Length": round(total_chars_target / total_words, 2),
        "Avg_Pred_Length": round(total_chars_pred / total_words, 2),
        "Avg_Levenshtein_Distance": round(np.mean(word_distances), 2),
        "Median_Levenshtein_Distance": round(np.median(word_distances), 2)
    }


# =====================================================================
# ERROR ANALYSIS
# =====================================================================
class ErrorAnalyzer:
    def __init__(self, output_dir="./error_analysis_gpt"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.errors = []
        self.correct_count = 0
    
    def add_error(self, prediction, target, image_path=None, confidence=None):
        self.errors.append({
            "prediction": prediction,
            "target": target,
            "image_path": image_path,
            "confidence": confidence,
            "levenshtein_distance": Levenshtein.distance(prediction, target),
            "length_diff": abs(len(prediction) - len(target))
        })
    
    def analyze(self):
        if not self.errors:
            print("No errors found!")
            return
        
        # Group by distance
        by_distance = defaultdict(list)
        for err in self.errors:
            by_distance[err["levenshtein_distance"]].append(err)
        
        # Find common character substitutions
        common_errors = defaultdict(int)
        char_confusions = defaultdict(lambda: defaultdict(int))
        
        for err in self.errors:
            pred = err["prediction"].lower()
            target = err["target"].lower()
            
            # Align strings for character-level analysis
            max_len = max(len(pred), len(target))
            for i in range(max_len):
                if i < len(target) and i < len(pred):
                    if pred[i] != target[i]:
                        common_errors[f"{target[i]}->{pred[i]}"] += 1
                        char_confusions[target[i]][pred[i]] += 1
                elif i < len(target):
                    common_errors[f"{target[i]}->[missing]"] += 1
                elif i < len(pred):
                    common_errors[f"[extra]->{pred[i]}"] += 1
        
        # Find length-based patterns
        length_errors = defaultdict(int)
        for err in self.errors:
            length_errors[err["length_diff"]] += 1
        
        # Save analysis
        report = {
            "total_errors": len(self.errors),
            "error_rate": (len(self.errors) / (len(self.errors) + self.correct_count)) * 100 if self.correct_count > 0 else 100,
            "errors_by_distance": {k: len(v) for k, v in by_distance.items()},
            "top_substitutions": dict(sorted(common_errors.items(), key=lambda x: x[1], reverse=True)[:20]),
            "length_error_distribution": dict(length_errors),
            "sample_errors": self.errors[:100]  # First 100 errors
        }
        
        with open(os.path.join(self.output_dir, "error_analysis.json"), 'w') as f:
            json.dump(report, f, indent=2)
        
        print(f"\nError analysis saved to {self.output_dir}/error_analysis.json")
        print(f"Total errors: {len(self.errors)}")
        print(f"Error rate: {report['error_rate']:.2f}%")
        print(f"\nTop 10 character substitutions:")
        for sub, count in list(report["top_substitutions"].items())[:10]:
            print(f"  {sub}: {count} times")
        
        return report


# =====================================================================
# INFERENCE FUNCTION WITH BEAM SEARCH
# =====================================================================
@torch.no_grad()
def run_inference(model, test_loader, tokenizer, device, 
                 use_beam_search=True, beam_size=5):
    """Run inference on test set with optional beam search"""
    model.eval()
    all_predictions = []
    all_targets = []
    all_image_paths = []
    all_confidence_scores = []
    
    for images, texts, img_paths in tqdm(test_loader, desc="Running inference"):
        images = images.to(device)
        
        if use_beam_search:
            # Use beam search for better accuracy
            tokens = model.generate(
                images,
                max_length=25,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                beam_size=beam_size
            )
            
            # For beam search, we don't have easy confidence scores
            # but we can compute them separately if needed
            confidence = [None] * len(tokens)
        else:
            # Use greedy decoding for speed
            tokens = model.generate(
                images,
                max_length=25,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                beam_size=1  # Greedy
            )
            confidence = [None] * len(tokens)
        
        # Decode predictions
        for i in range(len(tokens)):
            pred_text = tokenizer.decode(tokens[i], skip_special_tokens=True)
            all_predictions.append(pred_text)
            all_targets.append(texts[i])
            all_image_paths.append(img_paths[i])
            all_confidence_scores.append(confidence[i] if confidence else None)
    
    return all_predictions, all_targets, all_image_paths, all_confidence_scores


# =====================================================================
# CONFIDENCE CALCULATION (Optional)
# =====================================================================
def calculate_confidence(model, images, tokens, tokenizer):
    """Calculate confidence scores for predictions"""
    # This is a simplified confidence calculation
    # You can extend this to use log probabilities from beam search
    confidences = []
    for i in range(len(tokens)):
        # Simple confidence: length-normalized token probability
        # For production, extract from beam search scores
        confidences.append(0.5)  # Placeholder
    return confidences


# =====================================================================
# VISUALIZATION AND SAVING RESULTS
# =====================================================================
def save_results(predictions, targets, image_paths, metrics, output_dir):
    """Save test results to files"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save metrics
    with open(os.path.join(output_dir, "test_metrics.json"), 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Save all predictions
    with open(os.path.join(output_dir, "all_predictions.txt"), 'w') as f:
        f.write(f"{'Prediction':<40} {'Ground Truth':<40} {'Correct':<10} {'Image Path'}\n")
        f.write("=" * 120 + "\n")
        for pred, target, path in zip(predictions, targets, image_paths):
            correct = "✓" if pred.strip().lower() == target.strip().lower() else "✗"
            f.write(f"{pred:<40} {target:<40} {correct:<10} {path}\n")
    
    # Save correct predictions
    correct_preds = [(pred, target, path) for pred, target, path 
                     in zip(predictions, targets, image_paths) 
                     if pred.strip().lower() == target.strip().lower()]
    
    with open(os.path.join(output_dir, "correct_predictions.txt"), 'w') as f:
        f.write(f"Total Correct: {len(correct_preds)}/{len(predictions)}\n")
        f.write("=" * 80 + "\n\n")
        for pred, target, path in correct_preds[:200]:  # First 200 correct
            f.write(f"Pred: {pred} | Target: {target}\n")
            f.write(f"Path: {path}\n\n")
    
    # Save error cases
    errors = [(pred, target, path) for pred, target, path 
              in zip(predictions, targets, image_paths) 
              if pred.strip().lower() != target.strip().lower()]
    
    if errors:
        with open(os.path.join(output_dir, "error_cases.txt"), 'w') as f:
            f.write(f"Total Errors: {len(errors)}/{len(predictions)}\n")
            f.write(f"Error Rate: {(len(errors)/len(predictions))*100:.2f}%\n")
            f.write("=" * 80 + "\n\n")
            
            for pred, target, path in errors[:200]:  # First 200 errors
                dist = Levenshtein.distance(pred, target)
                f.write(f"Image: {path}\n")
                f.write(f"Predicted: {pred}\n")
                f.write(f"Target: {target}\n")
                f.write(f"Levenshtein Distance: {dist}\n")
                f.write("-" * 60 + "\n")
        
        # Also save as CSV for easier analysis
        import csv
        with open(os.path.join(output_dir, "errors.csv"), 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Prediction", "Target", "Distance", "Image_Path"])
            for pred, target, path in errors:
                dist = Levenshtein.distance(pred, target)
                writer.writerow([pred, target, dist, path])
    
    print(f"\nResults saved to: {output_dir}")
    print(f"  - test_metrics.json: Detailed metrics")
    print(f"  - all_predictions.txt: All predictions")
    print(f"  - correct_predictions.txt: Correct predictions")
    print(f"  - error_cases.txt: Error cases")
    if errors:
        print(f"  - errors.csv: Error cases in CSV format")


# =====================================================================
# PER-CHARACTER ACCURACY ANALYSIS
# =====================================================================
def analyze_character_accuracy(predictions, targets, output_dir):
    """Analyze accuracy per character"""
    char_stats = defaultdict(lambda: {"correct": 0, "total": 0, "confusions": defaultdict(int)})
    
    for pred, target in zip(predictions, targets):
        pred = pred.strip().lower()
        target = target.strip().lower()
        
        # Align sequences (simple length-based alignment)
        max_len = max(len(pred), len(target))
        for i in range(max_len):
            if i < len(target):
                target_char = target[i]
                char_stats[target_char]["total"] += 1
                
                if i < len(pred) and pred[i] == target_char:
                    char_stats[target_char]["correct"] += 1
                elif i < len(pred):
                    char_stats[target_char]["confusions"][pred[i]] += 1
    
    # Calculate accuracy per character
    char_accuracy = {}
    for char, stats in char_stats.items():
        acc = (stats["correct"] / stats["total"]) * 100 if stats["total"] > 0 else 0
        char_accuracy[char] = acc
    
    # Save results
    with open(os.path.join(output_dir, "char_accuracy.json"), 'w') as f:
        json.dump({
            "char_accuracy": {k: round(v, 2) for k, v in char_accuracy.items()},
            "total_chars": len(char_stats)
        }, f, indent=2)
    
    # Print worst-performing characters
    worst_chars = sorted(char_accuracy.items(), key=lambda x: x[1])[:15]
    print("\n" + "=" * 60)
    print("WORST PERFORMING CHARACTERS")
    print("=" * 60)
    for char, acc in worst_chars:
        if char.isalpha():
            print(f"  '{char}': {acc:.1f}% accuracy")
    
    return char_accuracy


# =====================================================================
# MAIN TEST FUNCTION
# =====================================================================
def test_model(checkpoint_path=None, use_beam_search=True, beam_size=5):
    """Main test function with beam search support"""
    
    # Load tokenizer
    print("=" * 60)
    print("TESTING GPT-2 BASED HTR MODEL")
    print("=" * 60)
    print("\nLoading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    print(f"BOS: {tokenizer.bos_token_id} | EOS: {tokenizer.eos_token_id} | PAD: {tokenizer.pad_token_id}")
    
    # Create test dataloader
    print("\nCreating test dataloader...")
    test_loader = get_test_dataloader(
        BASE_DATA_DIR, 
        WORDS_TXT_FILE, 
        tokenizer, 
        batch_size=TEST_CONFIG["batch_size"],
        img_height=IMG_HEIGHT,
        img_width=IMG_WIDTH
    )
    
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")
    
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    
    # Load checkpoint
    checkpoint_path = checkpoint_path or TEST_CONFIG["checkpoint_path"]
    if os.path.exists(checkpoint_path):
        print(f"\nLoading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Checkpoint info:")
        print(f"  - Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"  - Best WER: {checkpoint.get('best_wer', 'N/A')}%")
        print(f"  - CER at save: {checkpoint.get('cer', 'N/A')}%")
        if 'img_height' in checkpoint:
            print(f"  - Image size: {checkpoint['img_height']}x{checkpoint['img_width']}")
    else:
        print(f"\nWarning: Checkpoint not found at {checkpoint_path}")
        print("Using randomly initialized model")
        return None, None, None
    
    # Run inference
    print(f"\nRunning inference with {'BEAM SEARCH' if use_beam_search else 'GREEDY'} decoding...")
    print(f"Beam size: {beam_size if use_beam_search else 1}")
    
    predictions, targets, image_paths, confidences = run_inference(
        model, test_loader, tokenizer, device, 
        use_beam_search=use_beam_search, 
        beam_size=beam_size
    )
    
    # Calculate metrics
    print("\nCalculating metrics...")
    metrics = calculate_detailed_metrics(predictions, targets)
    
    # Print results
    print("\n" + "=" * 60)
    print("TEST RESULTS")
    print("=" * 60)
    print(f"Total test samples: {metrics['Total_Samples']}")
    print(f"\nCharacter Error Rate (CER): {metrics['CER']:.2f}%")
    print(f"Character Error Rate (standard): {metrics['CER_standard']:.2f}%")
    print(f"Word Error Rate (WER): {metrics['WER']:.2f}%")
    print(f"Word Accuracy: {metrics['Word_Accuracy']:.2f}%")
    print(f"\nCorrect words: {metrics['Correct_Words']}/{metrics['Total_Samples']}")
    print(f"Avg target length: {metrics['Avg_Target_Length']} chars")
    print(f"Avg prediction length: {metrics['Avg_Pred_Length']} chars")
    print(f"Avg Levenshtein distance: {metrics['Avg_Levenshtein_Distance']}")
    print("=" * 60)
    
    # Save results
    output_dir = TEST_CONFIG["output_dir"]
    if use_beam_search:
        output_dir = f"{output_dir}_beam{beam_size}"
    else:
        output_dir = f"{output_dir}_greedy"
    
    save_results(predictions, targets, image_paths, metrics, output_dir)
    
    # Error analysis
    print("\nPerforming error analysis...")
    analyzer = ErrorAnalyzer(output_dir=os.path.join(output_dir, "error_analysis"))
    analyzer.correct_count = metrics['Correct_Words']
    
    for pred, target, path in zip(predictions, targets, image_paths):
        if pred.strip().lower() != target.strip().lower():
            analyzer.add_error(pred, target, path)
    
    analyzer.analyze()
    
    # Character-level analysis
    print("\nAnalyzing character-level accuracy...")
    char_accuracy = analyze_character_accuracy(predictions, targets, output_dir)
    
    # Generate summary report
    summary = {
        "test_config": TEST_CONFIG,
        "results": metrics,
        "decoding_method": "beam_search" if use_beam_search else "greedy",
        "beam_size": beam_size if use_beam_search else 1,
        "checkpoint": checkpoint_path
    }
    
    with open(os.path.join(output_dir, "test_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n✅ Test complete! Summary saved to {output_dir}/test_summary.json")
    
    return metrics, predictions, targets


# =====================================================================
# COMPARISON FUNCTION (Greedy vs Beam Search)
# =====================================================================
def compare_decoding_methods():
    """Compare greedy vs beam search decoding"""
    print("\n" + "=" * 70)
    print("COMPARING GREEDY VS BEAM SEARCH DECODING")
    print("=" * 70)
    
    # Test with greedy
    print("\n1. Running greedy decoding (beam_size=1)...")
    metrics_greedy, _, _ = test_model(
        use_beam_search=False,
        beam_size=1
    )
    
    # Test with beam search
    print("\n2. Running beam search decoding (beam_size=5)...")
    metrics_beam, _, _ = test_model(
        use_beam_search=True,
        beam_size=5
    )
    
    # Compare results
    print("\n" + "=" * 70)
    print("COMPARISON RESULTS")
    print("=" * 70)
    print(f"{'Metric':<20} {'Greedy':>15} {'Beam Search':>15} {'Improvement':>15}")
    print("-" * 70)
    
    cer_improvement = metrics_greedy['CER'] - metrics_beam['CER']
    wer_improvement = metrics_greedy['WER'] - metrics_beam['WER']
    acc_improvement = metrics_beam['Word_Accuracy'] - metrics_greedy['Word_Accuracy']
    
    print(f"{'CER (%)':<20} {metrics_greedy['CER']:>15.2f} {metrics_beam['CER']:>15.2f} {cer_improvement:>+15.2f}")
    print(f"{'WER (%)':<20} {metrics_greedy['WER']:>15.2f} {metrics_beam['WER']:>15.2f} {wer_improvement:>+15.2f}")
    print(f"{'Word Acc (%)':<20} {metrics_greedy['Word_Accuracy']:>15.2f} {metrics_beam['Word_Accuracy']:>15.2f} {acc_improvement:>+15.2f}")
    print("=" * 70)
    
    # Save comparison
    comparison = {
        "greedy": metrics_greedy,
        "beam_search": metrics_beam,
        "improvements": {
            "CER": cer_improvement,
            "WER": wer_improvement,
            "Word_Accuracy": acc_improvement
        }
    }
    
    output_dir = TEST_CONFIG["output_dir"]
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "decoding_comparison.json"), 'w') as f:
        json.dump(comparison, f, indent=2)
    
    print(f"\nComparison saved to {output_dir}/decoding_comparison.json")


# =====================================================================
# RUN TESTS
# =====================================================================
if __name__ == "__main__":
    # Make sure the model classes are imported or defined
    # If running in same notebook, they should already be in memory
    
    # Option 1: Run single test with beam search
    # print("Running test with beam search (recommended for final evaluation)...")
    # metrics, predictions, targets = test_model(
    #     use_beam_search=True,
    #     beam_size=TEST_CONFIG["beam_size"]
    # )
    
    # Option 2: Compare greedy vs beam search (uncomment to run)
    # print("\n" + "="*70)
    # print("OPTIONAL: Comparing decoding methods")
    # print("="*70)
    # compare_decoding_methods()
    
    # Option 3: Test with different beam sizes (uncomment to run)
    for beam_size in [3, 5, 7]:
        print(f"\nTesting with beam_size={beam_size}")
        test_model(use_beam_search=True, beam_size=beam_size)


Testing with beam_size=3
TESTING GPT-2 BASED HTR MODEL

Loading tokenizer...
Vocabulary size: 50265
BOS: 0 | EOS: 2 | PAD: 1

Creating test dataloader...
Test samples loaded: 3830

Using device: cuda
Loading GPT-2 pretrained weights...
  Loaded: 147 weight tensors
  Missing (new layers): 98
GPT-2 weights loaded successfully.
  Vocab: 50257 pretrained + 8 new tokens
  Cross-attention layers: randomly initialized

Loading checkpoint: /home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth
Checkpoint info:
  - Epoch: 42
  - Best WER: 22.2396%
  - CER at save: 14.1156%
  - Image size: 96x384

Running inference with BEAM SEARCH decoding...
Beam size: 3


Running inference: 100%|██████████████████████| 120/120 [53:44<00:00, 26.87s/it]



Calculating metrics...

TEST RESULTS
Total test samples: 3830

Character Error Rate (CER): 1.40%
Character Error Rate (standard): 2.17%
Word Error Rate (WER): 2.17%
Word Accuracy: 97.83%

Correct words: 3747/3830
Avg target length: 4.08 chars
Avg prediction length: 4.08 chars
Avg Levenshtein distance: 0.09

Results saved to: ./test_results_gpt_beam3
  - test_metrics.json: Detailed metrics
  - all_predictions.txt: All predictions
  - correct_predictions.txt: Correct predictions
  - error_cases.txt: Error cases
  - errors.csv: Error cases in CSV format

Performing error analysis...

Error analysis saved to ./test_results_gpt_beam3/error_analysis/error_analysis.json
Total errors: 83
Error rate: 2.17%

Top 10 character substitutions:
  e->[missing]: 8 times
  [extra]->e: 7 times
  [extra]->y: 6 times
  a->e: 6 times
  a->o: 6 times
  s->e: 6 times
  n->r: 6 times
  y->[missing]: 6 times
  o->a: 5 times
  l->t: 5 times

Analyzing character-level accuracy...

WORST PERFORMING CHARACTERS
  '

Running inference: 100%|████████████████████| 120/120 [1:29:30<00:00, 44.75s/it]



Calculating metrics...

TEST RESULTS
Total test samples: 3830

Character Error Rate (CER): 1.40%
Character Error Rate (standard): 2.16%
Word Error Rate (WER): 2.17%
Word Accuracy: 97.83%

Correct words: 3747/3830
Avg target length: 4.08 chars
Avg prediction length: 4.08 chars
Avg Levenshtein distance: 0.09

Results saved to: ./test_results_gpt_beam5
  - test_metrics.json: Detailed metrics
  - all_predictions.txt: All predictions
  - correct_predictions.txt: Correct predictions
  - error_cases.txt: Error cases
  - errors.csv: Error cases in CSV format

Performing error analysis...

Error analysis saved to ./test_results_gpt_beam5/error_analysis/error_analysis.json
Total errors: 83
Error rate: 2.17%

Top 10 character substitutions:
  e->[missing]: 9 times
  [extra]->e: 7 times
  [extra]->y: 6 times
  a->e: 6 times
  a->o: 6 times
  s->e: 6 times
  n->r: 6 times
  y->[missing]: 6 times
  o->a: 5 times
  l->t: 5 times

Analyzing character-level accuracy...

WORST PERFORMING CHARACTERS
  '

Running inference: 100%|████████████████████| 120/120 [2:04:33<00:00, 62.28s/it]


Calculating metrics...

TEST RESULTS
Total test samples: 3830

Character Error Rate (CER): 1.40%
Character Error Rate (standard): 2.16%
Word Error Rate (WER): 2.17%
Word Accuracy: 97.83%

Correct words: 3747/3830
Avg target length: 4.08 chars
Avg prediction length: 4.08 chars
Avg Levenshtein distance: 0.09

Results saved to: ./test_results_gpt_beam7
  - test_metrics.json: Detailed metrics
  - all_predictions.txt: All predictions
  - correct_predictions.txt: Correct predictions
  - error_cases.txt: Error cases
  - errors.csv: Error cases in CSV format

Performing error analysis...

Error analysis saved to ./test_results_gpt_beam7/error_analysis/error_analysis.json
Total errors: 83
Error rate: 2.17%

Top 10 character substitutions:
  e->[missing]: 9 times
  [extra]->e: 7 times
  [extra]->y: 6 times
  a->e: 6 times
  a->o: 6 times
  s->e: 6 times
  n->r: 6 times
  y->[missing]: 6 times
  o->a: 5 times
  l->t: 5 times

Analyzing character-level accuracy...

WORST PERFORMING CHARACTERS
  '

In [3]:
# =====================================================================
# TESTING CODE FOR GPT-2 BASED HTR MODEL - GREEDY DECODING ONLY
# =====================================================================

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer
from PIL import Image
import Levenshtein
from tqdm import tqdm
import numpy as np
from collections import defaultdict

# =====================================================================
# TEST CONFIGURATION
# =====================================================================
TEST_CONFIG = {
    "checkpoint_path": "/home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth",
    "batch_size": 32,  # Can be larger for testing since no gradients
    "num_workers": 4,
    "save_predictions": True,
    "output_dir": "./test_results_gpt_greedy",
    "test_ratio": 0.1,  # Use 10% of data for testing
}

# =====================================================================
# TEST DATASET (Replicates validation dataset but with test split)
# =====================================================================
class IAMTestDataset:
    def __init__(self, data_dir, words_txt_path, processor, transform=None, 
                 max_target_length=25, test_ratio=0.1):
        self.data_dir = data_dir
        self.processor = processor
        self.transform = transform
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_test_set(words_txt_path, test_ratio)
    
    def _parse_test_set(self, words_txt_path, test_ratio):
        if not os.path.exists(words_txt_path):
            print(f"Error: {words_txt_path} not found.")
            return
        
        all_samples = []
        with open(words_txt_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9 or parts[1] != "ok":
                    continue
                
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                word_id = parts[0]
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    all_samples.append((img_path, transcription))
        
        # Use last test_ratio as test set (consistent with training)
        test_size = int(len(all_samples) * test_ratio)
        self.samples = all_samples[-test_size:] if test_size > 0 else all_samples
        print(f"Test samples loaded: {len(self.samples)}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (384, 96), color=255)
        
        if self.transform:
            image = self.transform(image)
        
        return image, text, img_path


def get_test_dataloader(data_dir, words_txt_path, processor, 
                       batch_size=32, img_height=96, img_width=384):
    """Create test dataloader"""
    test_transform = T.Compose([
        T.Resize((img_height, img_width)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], 
                   std=[0.229, 0.224, 0.225])
    ])
    
    test_dataset = IAMTestDataset(
        data_dir, words_txt_path, processor, 
        transform=test_transform, 
        test_ratio=TEST_CONFIG["test_ratio"]
    )
    
    def collate_fn(batch):
        images = torch.stack([b[0] for b in batch])
        texts = [b[1] for b in batch]
        paths = [b[2] for b in batch]
        return images, texts, paths
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        collate_fn=collate_fn
    )
    return test_loader


# =====================================================================
# ENHANCED METRICS
# =====================================================================
def calculate_detailed_metrics(predictions, targets):
    """Calculate detailed metrics including character-level and word-level stats"""
    if not targets:
        return {}
    
    total_chars_correct = 0
    total_chars_target = 0
    total_chars_pred = 0
    total_words_correct = 0
    total_words = len(targets)
    
    cer_per_sample = []
    wer_per_sample = []
    word_distances = []
    
    for pred, target in zip(predictions, targets):
        pred = pred.strip().lower()
        target = target.strip().lower()
        
        # Character-level
        dist = Levenshtein.distance(pred, target)
        target_len = len(target)
        if target_len > 0:
            cer = dist / target_len
            cer_per_sample.append(cer)
            total_chars_correct += (target_len - dist)
            total_chars_target += target_len
            total_chars_pred += len(pred)
        
        # Word-level
        word_distances.append(dist)
        if pred == target:
            total_words_correct += 1
            wer_per_sample.append(0.0)
        else:
            wer_per_sample.append(1.0)
    
    avg_cer = np.mean(cer_per_sample) * 100 if cer_per_sample else 0
    avg_wer = np.mean(wer_per_sample) * 100 if wer_per_sample else 0
    cer_standard = (total_chars_target - total_chars_correct) / total_chars_target * 100 if total_chars_target > 0 else 0
    
    return {
        "CER": round(avg_cer, 4),
        "CER_standard": round(cer_standard, 4),
        "WER": round(avg_wer, 4),
        "Word_Accuracy": round((total_words_correct / total_words) * 100, 4),
        "Total_Samples": total_words,
        "Correct_Words": total_words_correct,
        "Avg_Target_Length": round(total_chars_target / total_words, 2),
        "Avg_Pred_Length": round(total_chars_pred / total_words, 2),
        "Avg_Levenshtein_Distance": round(np.mean(word_distances), 2),
        "Median_Levenshtein_Distance": round(np.median(word_distances), 2)
    }


# =====================================================================
# ERROR ANALYSIS
# =====================================================================
class ErrorAnalyzer:
    def __init__(self, output_dir="./error_analysis_gpt"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.errors = []
        self.correct_count = 0
    
    def add_error(self, prediction, target, image_path=None, confidence=None):
        self.errors.append({
            "prediction": prediction,
            "target": target,
            "image_path": image_path,
            "confidence": confidence,
            "levenshtein_distance": Levenshtein.distance(prediction, target),
            "length_diff": abs(len(prediction) - len(target))
        })
    
    def analyze(self):
        if not self.errors:
            print("No errors found!")
            return
        
        # Group by distance
        by_distance = defaultdict(list)
        for err in self.errors:
            by_distance[err["levenshtein_distance"]].append(err)
        
        # Find common character substitutions
        common_errors = defaultdict(int)
        char_confusions = defaultdict(lambda: defaultdict(int))
        
        for err in self.errors:
            pred = err["prediction"].lower()
            target = err["target"].lower()
            
            # Align strings for character-level analysis
            max_len = max(len(pred), len(target))
            for i in range(max_len):
                if i < len(target) and i < len(pred):
                    if pred[i] != target[i]:
                        common_errors[f"{target[i]}->{pred[i]}"] += 1
                        char_confusions[target[i]][pred[i]] += 1
                elif i < len(target):
                    common_errors[f"{target[i]}->[missing]"] += 1
                elif i < len(pred):
                    common_errors[f"[extra]->{pred[i]}"] += 1
        
        # Find length-based patterns
        length_errors = defaultdict(int)
        for err in self.errors:
            length_errors[err["length_diff"]] += 1
        
        # Save analysis
        report = {
            "total_errors": len(self.errors),
            "error_rate": (len(self.errors) / (len(self.errors) + self.correct_count)) * 100 if self.correct_count > 0 else 100,
            "errors_by_distance": {k: len(v) for k, v in by_distance.items()},
            "top_substitutions": dict(sorted(common_errors.items(), key=lambda x: x[1], reverse=True)[:20]),
            "length_error_distribution": dict(length_errors),
            "sample_errors": self.errors[:100]  # First 100 errors
        }
        
        with open(os.path.join(self.output_dir, "error_analysis.json"), 'w') as f:
            json.dump(report, f, indent=2)
        
        print(f"\nError analysis saved to {self.output_dir}/error_analysis.json")
        print(f"Total errors: {len(self.errors)}")
        print(f"Error rate: {report['error_rate']:.2f}%")
        print(f"\nTop 10 character substitutions:")
        for sub, count in list(report["top_substitutions"].items())[:10]:
            print(f"  {sub}: {count} times")
        
        return report


# =====================================================================
# INFERENCE FUNCTION WITH GREEDY DECODING
# =====================================================================
@torch.no_grad()
def run_inference_greedy(model, test_loader, tokenizer, device):
    """Run inference on test set with greedy decoding (fastest)"""
    model.eval()
    all_predictions = []
    all_targets = []
    all_image_paths = []
    
    for images, texts, img_paths in tqdm(test_loader, desc="Running greedy inference"):
        images = images.to(device)
        
        # Use greedy decoding (beam_size=1)
        tokens = model.generate(
            images,
            max_length=25,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            beam_size=1  # Greedy decoding
        )
        
        # Decode predictions
        for i in range(len(tokens)):
            pred_text = tokenizer.decode(tokens[i], skip_special_tokens=True)
            all_predictions.append(pred_text)
            all_targets.append(texts[i])
            all_image_paths.append(img_paths[i])
    
    return all_predictions, all_targets, all_image_paths


# =====================================================================
# VISUALIZATION AND SAVING RESULTS
# =====================================================================
def save_results(predictions, targets, image_paths, metrics, output_dir):
    """Save test results to files"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save metrics
    with open(os.path.join(output_dir, "test_metrics.json"), 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Save all predictions
    with open(os.path.join(output_dir, "all_predictions.txt"), 'w') as f:
        f.write(f"{'Prediction':<40} {'Ground Truth':<40} {'Correct':<10}\n")
        f.write("=" * 90 + "\n")
        for pred, target in zip(predictions, targets):
            correct = "✓" if pred.strip().lower() == target.strip().lower() else "✗"
            f.write(f"{pred:<40} {target:<40} {correct:<10}\n")
    
    # Save correct predictions
    correct_preds = [(pred, target, path) for pred, target, path 
                     in zip(predictions, targets, image_paths) 
                     if pred.strip().lower() == target.strip().lower()]
    
    with open(os.path.join(output_dir, "correct_predictions.txt"), 'w') as f:
        f.write(f"Total Correct: {len(correct_preds)}/{len(predictions)}\n")
        f.write("=" * 80 + "\n\n")
        for pred, target, path in correct_preds[:200]:  # First 200 correct
            f.write(f"Pred: {pred} | Target: {target}\n")
            f.write(f"Path: {path}\n\n")
    
    # Save error cases
    errors = [(pred, target, path) for pred, target, path 
              in zip(predictions, targets, image_paths) 
              if pred.strip().lower() != target.strip().lower()]
    
    if errors:
        with open(os.path.join(output_dir, "error_cases.txt"), 'w') as f:
            f.write(f"Total Errors: {len(errors)}/{len(predictions)}\n")
            f.write(f"Error Rate: {(len(errors)/len(predictions))*100:.2f}%\n")
            f.write("=" * 80 + "\n\n")
            
            for pred, target, path in errors[:200]:  # First 200 errors
                dist = Levenshtein.distance(pred, target)
                f.write(f"Image: {path}\n")
                f.write(f"Predicted: {pred}\n")
                f.write(f"Target: {target}\n")
                f.write(f"Levenshtein Distance: {dist}\n")
                f.write("-" * 60 + "\n")
        
        # Also save as CSV for easier analysis
        import csv
        with open(os.path.join(output_dir, "errors.csv"), 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Prediction", "Target", "Distance", "Image_Path"])
            for pred, target, path in errors:
                dist = Levenshtein.distance(pred, target)
                writer.writerow([pred, target, dist, path])
    
    print(f"\nResults saved to: {output_dir}")
    print(f"  - test_metrics.json: Detailed metrics")
    print(f"  - all_predictions.txt: All predictions")
    print(f"  - correct_predictions.txt: Correct predictions")
    print(f"  - error_cases.txt: Error cases")
    if errors:
        print(f"  - errors.csv: Error cases in CSV format")


# =====================================================================
# PER-CHARACTER ACCURACY ANALYSIS
# =====================================================================
def analyze_character_accuracy(predictions, targets, output_dir):
    """Analyze accuracy per character"""
    char_stats = defaultdict(lambda: {"correct": 0, "total": 0, "confusions": defaultdict(int)})
    
    for pred, target in zip(predictions, targets):
        pred = pred.strip().lower()
        target = target.strip().lower()
        
        # Align sequences (simple length-based alignment)
        max_len = max(len(pred), len(target))
        for i in range(max_len):
            if i < len(target):
                target_char = target[i]
                char_stats[target_char]["total"] += 1
                
                if i < len(pred) and pred[i] == target_char:
                    char_stats[target_char]["correct"] += 1
                elif i < len(pred):
                    char_stats[target_char]["confusions"][pred[i]] += 1
    
    # Calculate accuracy per character
    char_accuracy = {}
    for char, stats in char_stats.items():
        acc = (stats["correct"] / stats["total"]) * 100 if stats["total"] > 0 else 0
        char_accuracy[char] = acc
    
    # Save results
    with open(os.path.join(output_dir, "char_accuracy.json"), 'w') as f:
        json.dump({
            "char_accuracy": {k: round(v, 2) for k, v in char_accuracy.items()},
            "total_chars": len(char_stats)
        }, f, indent=2)
    
    # Print worst-performing characters
    worst_chars = sorted(char_accuracy.items(), key=lambda x: x[1])[:15]
    print("\n" + "=" * 60)
    print("WORST PERFORMING CHARACTERS")
    print("=" * 60)
    for char, acc in worst_chars:
        if char.isalpha():
            print(f"  '{char}': {acc:.1f}% accuracy")
    
    return char_accuracy


# =====================================================================
# MAIN TEST FUNCTION - GREEDY DECODING ONLY
# =====================================================================
def test_model_greedy(checkpoint_path=None):
    """Main test function with greedy decoding only"""
    
    # Load tokenizer
    print("=" * 60)
    print("TESTING GPT-2 BASED HTR MODEL (GREEDY DECODING)")
    print("=" * 60)
    print("\nLoading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    print(f"BOS: {tokenizer.bos_token_id} | EOS: {tokenizer.eos_token_id} | PAD: {tokenizer.pad_token_id}")
    
    # Create test dataloader
    print("\nCreating test dataloader...")
    test_loader = get_test_dataloader(
        BASE_DATA_DIR, 
        WORDS_TXT_FILE, 
        tokenizer, 
        batch_size=TEST_CONFIG["batch_size"],
        img_height=IMG_HEIGHT,
        img_width=IMG_WIDTH
    )
    
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")
    
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    
    # Load checkpoint
    checkpoint_path = checkpoint_path or TEST_CONFIG["checkpoint_path"]
    if os.path.exists(checkpoint_path):
        print(f"\nLoading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Checkpoint info:")
        print(f"  - Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"  - Best WER: {checkpoint.get('best_wer', 'N/A')}%")
        print(f"  - CER at save: {checkpoint.get('cer', 'N/A')}%")
        if 'img_height' in checkpoint:
            print(f"  - Image size: {checkpoint['img_height']}x{checkpoint['img_width']}")
    else:
        print(f"\nWarning: Checkpoint not found at {checkpoint_path}")
        print("Using randomly initialized model")
        return None, None, None
    
    # Run inference with greedy decoding
    print(f"\nRunning inference with GREEDY decoding (fastest)...")
    
    predictions, targets, image_paths = run_inference_greedy(
        model, test_loader, tokenizer, device
    )
    
    # Calculate metrics
    print("\nCalculating metrics...")
    metrics = calculate_detailed_metrics(predictions, targets)
    
    # Print results
    print("\n" + "=" * 60)
    print("TEST RESULTS - GREEDY DECODING")
    print("=" * 60)
    print(f"Total test samples: {metrics['Total_Samples']}")
    print(f"\nCharacter Error Rate (CER): {metrics['CER']:.2f}%")
    print(f"Character Error Rate (standard): {metrics['CER_standard']:.2f}%")
    print(f"Word Error Rate (WER): {metrics['WER']:.2f}%")
    print(f"Word Accuracy: {metrics['Word_Accuracy']:.2f}%")
    print(f"\nCorrect words: {metrics['Correct_Words']}/{metrics['Total_Samples']}")
    print(f"Avg target length: {metrics['Avg_Target_Length']} chars")
    print(f"Avg prediction length: {metrics['Avg_Pred_Length']} chars")
    print(f"Avg Levenshtein distance: {metrics['Avg_Levenshtein_Distance']}")
    print("=" * 60)
    
    # Save results
    output_dir = TEST_CONFIG["output_dir"]
    
    save_results(predictions, targets, image_paths, metrics, output_dir)
    
    # Error analysis
    print("\nPerforming error analysis...")
    analyzer = ErrorAnalyzer(output_dir=os.path.join(output_dir, "error_analysis"))
    analyzer.correct_count = metrics['Correct_Words']
    
    for pred, target, path in zip(predictions, targets, image_paths):
        if pred.strip().lower() != target.strip().lower():
            analyzer.add_error(pred, target, path)
    
    analyzer.analyze()
    
    # Character-level analysis
    print("\nAnalyzing character-level accuracy...")
    char_accuracy = analyze_character_accuracy(predictions, targets, output_dir)
    
    # Generate summary report
    summary = {
        "test_config": TEST_CONFIG,
        "results": metrics,
        "decoding_method": "greedy",
        "beam_size": 1,
        "checkpoint": checkpoint_path
    }
    
    with open(os.path.join(output_dir, "test_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n✅ Test complete! Summary saved to {output_dir}/test_summary.json")
    
    return metrics, predictions, targets


# =====================================================================
# SAMPLE VISUALIZATION FUNCTION
# =====================================================================
def show_sample_predictions(predictions, targets, image_paths, num_samples=10):
    """Display sample predictions in a readable format"""
    print("\n" + "=" * 80)
    print(f"SAMPLE PREDICTIONS (First {num_samples} samples)")
    print("=" * 80)
    
    correct_count = 0
    for i in range(min(num_samples, len(predictions))):
        pred = predictions[i].strip()
        target = targets[i].strip()
        is_correct = pred.lower() == target.lower()
        correct_count += 1 if is_correct else 0
        status = "✓" if is_correct else "✗"
        
        print(f"\nSample {i+1}:")
        print(f"  Predicted: {pred}")
        print(f"  Target:    {target}")
        print(f"  Status:    {status}")
        if not is_correct:
            dist = Levenshtein.distance(pred.lower(), target.lower())
            print(f"  Distance:  {dist}")
    
    print(f"\nAccuracy on {num_samples} samples: {(correct_count/num_samples)*100:.1f}%")


# =====================================================================
# RUN TESTS
# =====================================================================
if __name__ == "__main__":
    # Run single test with greedy decoding
    print("Running test with greedy decoding (fast)...")
    metrics, predictions, targets = test_model_greedy()
    
    # Show sample predictions
    if predictions and targets:
        # Note: You'll need image_paths from the test function
        # For now, just show predictions without paths
        print("\n" + "=" * 80)
        print("SAMPLE PREDICTIONS")
        print("=" * 80)
        for i in range(min(20, len(predictions))):
            pred = predictions[i].strip()
            target = targets[i].strip()
            status = "✓" if pred.lower() == target.lower() else "✗"
            print(f"{status} Pred: {pred:<30} Target: {target}")

Running test with greedy decoding (fast)...
TESTING GPT-2 BASED HTR MODEL (GREEDY DECODING)

Loading tokenizer...
Vocabulary size: 50265
BOS: 0 | EOS: 2 | PAD: 1

Creating test dataloader...
Test samples loaded: 3830

Using device: cuda
Loading GPT-2 pretrained weights...
  Loaded: 147 weight tensors
  Missing (new layers): 98
GPT-2 weights loaded successfully.
  Vocab: 50257 pretrained + 8 new tokens
  Cross-attention layers: randomly initialized

Loading checkpoint: /home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth
Checkpoint info:
  - Epoch: 42
  - Best WER: 22.2396%
  - CER at save: 14.1156%
  - Image size: 96x384

Running inference with GREEDY decoding (fastest)...


Running greedy inference: 100%|███████████████| 120/120 [00:16<00:00,  7.08it/s]



Calculating metrics...

TEST RESULTS - GREEDY DECODING
Total test samples: 3830

Character Error Rate (CER): 1.45%
Character Error Rate (standard): 2.30%
Word Error Rate (WER): 2.22%
Word Accuracy: 97.78%

Correct words: 3745/3830
Avg target length: 4.08 chars
Avg prediction length: 4.08 chars
Avg Levenshtein distance: 0.09

Results saved to: ./test_results_gpt_greedy
  - test_metrics.json: Detailed metrics
  - all_predictions.txt: All predictions
  - correct_predictions.txt: Correct predictions
  - error_cases.txt: Error cases
  - errors.csv: Error cases in CSV format

Performing error analysis...

Error analysis saved to ./test_results_gpt_greedy/error_analysis/error_analysis.json
Total errors: 85
Error rate: 2.22%

Top 10 character substitutions:
  [extra]->e: 9 times
  a->e: 8 times
  e->[missing]: 8 times
  n->r: 8 times
  s->e: 7 times
  [extra]->s: 7 times
  e->r: 6 times
  l->e: 6 times
  [extra]->y: 5 times
  g->[missing]: 5 times

Analyzing character-level accuracy...

WORST

In [3]:
# =====================================================================
# TESTING CODE FOR GPT-2 BASED HTR MODEL - WITH AGENTIC VERIFICATION
# =====================================================================

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer
from PIL import Image
import Levenshtein
from tqdm import tqdm
import numpy as np
from collections import defaultdict

# =====================================================================
# TEST CONFIGURATION
# =====================================================================
TEST_CONFIG = {
    "checkpoint_path": "/home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth",
    "batch_size": 32,
    "num_workers": 4,
    "save_predictions": True,
    "output_dir": "./test_results_gpt_greedy_with_verification",
    "test_ratio": 0.1,
}

# =====================================================================
# AGENTIC VERIFICATION MODULE (Same as in training)
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = {}
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = defaultdict(int)
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] += 1
        self.lexicon = dict(freq)
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Agentic Verifier Lexicon: {len(self.lexicon)} words | Max freq: {self.freq_max}")

    def verify_and_correct(self, text_output, top_logprob=None, logprob_threshold=-0.3):
        cleaned = text_output.strip().lower()

        # Skip: empty, already valid, very short, has digits
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        # Skip proper nouns — capitalized words are likely correct
        if text_output and text_output[0].isupper() and len(text_output) > 1:
            if text_output[1:].islower():
                if cleaned in self.lexicon:
                    return text_output

        # Confidence gate
        if top_logprob is not None:
            avg_lp = sum(top_logprob) / max(len(top_logprob), 1)
            if avg_lp > logprob_threshold:
                return text_output

        best_candidate = None
        best_score = -float('inf')
        target_len = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue
            freq_score = freq / self.freq_max
            score = freq_score - (dist * 0.3)
            if score > best_score:
                best_score = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        # Preserve casing
        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# TEST DATASET
# =====================================================================
class IAMTestDataset:
    def __init__(self, data_dir, words_txt_path, processor, transform=None, 
                 max_target_length=25, test_ratio=0.1):
        self.data_dir = data_dir
        self.processor = processor
        self.transform = transform
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_test_set(words_txt_path, test_ratio)
    
    def _parse_test_set(self, words_txt_path, test_ratio):
        if not os.path.exists(words_txt_path):
            print(f"Error: {words_txt_path} not found.")
            return
        
        all_samples = []
        with open(words_txt_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9 or parts[1] != "ok":
                    continue
                
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                word_id = parts[0]
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    all_samples.append((img_path, transcription))
        
        test_size = int(len(all_samples) * test_ratio)
        self.samples = all_samples[-test_size:] if test_size > 0 else all_samples
        print(f"Test samples loaded: {len(self.samples)}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (384, 96), color=255)
        
        if self.transform:
            image = self.transform(image)
        
        return image, text, img_path


def get_test_dataloader(data_dir, words_txt_path, processor, 
                       batch_size=32, img_height=96, img_width=384):
    test_transform = T.Compose([
        T.Resize((img_height, img_width)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], 
                   std=[0.229, 0.224, 0.225])
    ])
    
    test_dataset = IAMTestDataset(
        data_dir, words_txt_path, processor, 
        transform=test_transform, 
        test_ratio=TEST_CONFIG["test_ratio"]
    )
    
    def collate_fn(batch):
        images = torch.stack([b[0] for b in batch])
        texts = [b[1] for b in batch]
        paths = [b[2] for b in batch]
        return images, texts, paths
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        collate_fn=collate_fn
    )
    return test_loader


# =====================================================================
# METRICS
# =====================================================================
def calculate_detailed_metrics(predictions, targets):
    if not targets:
        return {}
    
    total_chars_correct = 0
    total_chars_target = 0
    total_chars_pred = 0
    total_words_correct = 0
    total_words = len(targets)
    
    cer_per_sample = []
    wer_per_sample = []
    word_distances = []
    
    for pred, target in zip(predictions, targets):
        pred = pred.strip().lower()
        target = target.strip().lower()
        
        dist = Levenshtein.distance(pred, target)
        target_len = len(target)
        if target_len > 0:
            cer = dist / target_len
            cer_per_sample.append(cer)
            total_chars_correct += (target_len - dist)
            total_chars_target += target_len
            total_chars_pred += len(pred)
        
        word_distances.append(dist)
        if pred == target:
            total_words_correct += 1
            wer_per_sample.append(0.0)
        else:
            wer_per_sample.append(1.0)
    
    avg_cer = np.mean(cer_per_sample) * 100 if cer_per_sample else 0
    avg_wer = np.mean(wer_per_sample) * 100 if wer_per_sample else 0
    cer_standard = (total_chars_target - total_chars_correct) / total_chars_target * 100 if total_chars_target > 0 else 0
    
    return {
        "CER": round(avg_cer, 4),
        "CER_standard": round(cer_standard, 4),
        "WER": round(avg_wer, 4),
        "Word_Accuracy": round((total_words_correct / total_words) * 100, 4),
        "Total_Samples": total_words,
        "Correct_Words": total_words_correct,
        "Avg_Target_Length": round(total_chars_target / total_words, 2),
        "Avg_Pred_Length": round(total_chars_pred / total_words, 2),
        "Avg_Levenshtein_Distance": round(np.mean(word_distances), 2),
        "Median_Levenshtein_Distance": round(np.median(word_distances), 2)
    }


# =====================================================================
# INFERENCE WITH AGENTIC VERIFICATION
# =====================================================================
@torch.no_grad()
def run_inference_with_verification(model, test_loader, tokenizer, device, agent_verifier):
    """Run inference with agentic verification applied"""
    model.eval()
    all_predictions_raw = []
    all_predictions_verified = []
    all_targets = []
    all_image_paths = []
    verification_stats = {"total_corrections": 0, "total_samples": 0}
    
    for images, texts, img_paths in tqdm(test_loader, desc="Running inference with verification"):
        images = images.to(device)
        
        # Greedy decoding
        tokens = model.generate(
            images,
            max_length=25,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            beam_size=1
        )
        
        for i in range(len(tokens)):
            raw_pred = tokenizer.decode(tokens[i], skip_special_tokens=True)
            # Apply agentic verification
            verified_pred = agent_verifier.verify_and_correct(raw_pred)
            
            all_predictions_raw.append(raw_pred)
            all_predictions_verified.append(verified_pred)
            all_targets.append(texts[i])
            all_image_paths.append(img_paths[i])
            
            if raw_pred != verified_pred:
                verification_stats["total_corrections"] += 1
            verification_stats["total_samples"] += 1
    
    print(f"\n📊 Agentic Verification Stats:")
    print(f"  - Total corrections made: {verification_stats['total_corrections']}/{verification_stats['total_samples']}")
    print(f"  - Correction rate: {(verification_stats['total_corrections']/verification_stats['total_samples'])*100:.2f}%")
    
    return all_predictions_raw, all_predictions_verified, all_targets, all_image_paths, verification_stats


# =====================================================================
# SAVE RESULTS
# =====================================================================
def save_results_with_comparison(predictions_raw, predictions_verified, targets, 
                                 image_paths, metrics_raw, metrics_verified, 
                                 verification_stats, output_dir):
    """Save results showing before/after verification"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save metrics
    with open(os.path.join(output_dir, "test_metrics_raw.json"), 'w') as f:
        json.dump(metrics_raw, f, indent=2)
    
    with open(os.path.join(output_dir, "test_metrics_verified.json"), 'w') as f:
        json.dump(metrics_verified, f, indent=2)
    
    # Save comparison summary
    comparison = {
        "without_verification": metrics_raw,
        "with_verification": metrics_verified,
        "improvements": {
            "CER_improvement": metrics_raw['CER'] - metrics_verified['CER'],
            "WER_improvement": metrics_raw['WER'] - metrics_verified['WER'],
            "Word_Accuracy_improvement": metrics_verified['Word_Accuracy'] - metrics_raw['Word_Accuracy']
        },
        "verification_stats": verification_stats
    }
    
    with open(os.path.join(output_dir, "verification_comparison.json"), 'w') as f:
        json.dump(comparison, f, indent=2)
    
    # Save detailed predictions with comparison
    with open(os.path.join(output_dir, "predictions_comparison.txt"), 'w') as f:
        f.write(f"{'Raw Prediction':<35} {'Verified':<35} {'Target':<35} {'Changed':<10}\n")
        f.write("=" * 115 + "\n")
        for raw, verified, target in zip(predictions_raw, predictions_verified, targets):
            changed = "✓" if raw != verified else "-"
            f.write(f"{raw:<35} {verified:<35} {target:<35} {changed:<10}\n")
    
    # Save cases where verification helped
    helped_cases = [(raw, verified, target, path) for raw, verified, target, path 
                    in zip(predictions_raw, predictions_verified, targets, image_paths) 
                    if raw != verified and verified.strip().lower() == target.strip().lower()]
    
    if helped_cases:
        with open(os.path.join(output_dir, "verification_helped.txt"), 'w') as f:
            f.write(f"Verification helped fix {len(helped_cases)} cases!\n")
            f.write("=" * 80 + "\n\n")
            for raw, verified, target, path in helped_cases[:100]:
                f.write(f"Image: {path}\n")
                f.write(f"Raw: {raw}\n")
                f.write(f"Verified: {verified}\n")
                f.write(f"Target: {target}\n")
                f.write("-" * 60 + "\n")
    
    # Save cases where verification made it worse
    worse_cases = [(raw, verified, target) for raw, verified, target 
                   in zip(predictions_raw, predictions_verified, targets) 
                   if raw != verified and verified.strip().lower() != target.strip().lower() 
                   and Levenshtein.distance(verified, target) > Levenshtein.distance(raw, target)]
    
    if worse_cases:
        with open(os.path.join(output_dir, "verification_made_worse.txt"), 'w') as f:
            f.write(f"⚠️ Verification made {len(worse_cases)} cases worse!\n")
            f.write("=" * 80 + "\n\n")
            for raw, verified, target in worse_cases[:50]:
                raw_dist = Levenshtein.distance(raw, target)
                ver_dist = Levenshtein.distance(verified, target)
                f.write(f"Raw: {raw} (dist={raw_dist})\n")
                f.write(f"Verified: {verified} (dist={ver_dist})\n")
                f.write(f"Target: {target}\n")
                f.write("-" * 60 + "\n")
    
    print(f"\n✅ Results saved to: {output_dir}")
    print(f"  - verification_comparison.json: Before/after metrics")
    print(f"  - predictions_comparison.txt: All predictions with comparison")
    if helped_cases:
        print(f"  - verification_helped.txt: {len(helped_cases)} cases fixed")
    if worse_cases:
        print(f"  - verification_made_worse.txt: {len(worse_cases)} cases broken")


# =====================================================================
# MAIN TEST FUNCTION WITH VERIFICATION
# =====================================================================
def test_model_with_verification(checkpoint_path=None):
    """Main test function with agentic verification"""
    
    print("=" * 60)
    print("TESTING GPT-2 BASED HTR MODEL WITH AGENTIC VERIFICATION")
    print("=" * 60)
    
    # Load tokenizer
    print("\nLoading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id}")
    
    # Initialize agent verifier
    print("\nInitializing Agentic Verification Module...")
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    # Create test dataloader
    print("\nCreating test dataloader...")
    test_loader = get_test_dataloader(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, 
        batch_size=TEST_CONFIG["batch_size"],
        img_height=IMG_HEIGHT,
        img_width=IMG_WIDTH
    )
    
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")
    
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    
    checkpoint_path = checkpoint_path or TEST_CONFIG["checkpoint_path"]
    if os.path.exists(checkpoint_path):
        print(f"\nLoading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"  - Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"  - Best WER: {checkpoint.get('best_wer', 'N/A')}%")
    else:
        print(f"\nCheckpoint not found: {checkpoint_path}")
        return None
    
    # Run inference with verification
    print("\nRunning inference with agentic verification...")
    predictions_raw, predictions_verified, targets, image_paths, ver_stats = \
        run_inference_with_verification(model, test_loader, tokenizer, device, agent_verifier)
    
    # Calculate metrics (without verification)
    print("\nCalculating metrics (without verification)...")
    metrics_raw = calculate_detailed_metrics(predictions_raw, targets)
    
    # Calculate metrics (with verification)
    print("Calculating metrics (WITH verification)...")
    metrics_verified = calculate_detailed_metrics(predictions_verified, targets)
    
    # Print comparison
    print("\n" + "=" * 70)
    print("TEST RESULTS - WITH VS WITHOUT VERIFICATION")
    print("=" * 70)
    print(f"\n{'Metric':<25} {'Without Verif':>15} {'With Verif':>15} {'Improvement':>15}")
    print("-" * 70)
    
    cer_imp = metrics_raw['CER'] - metrics_verified['CER']
    wer_imp = metrics_raw['WER'] - metrics_verified['WER']
    acc_imp = metrics_verified['Word_Accuracy'] - metrics_raw['Word_Accuracy']
    
    print(f"{'CER (%)':<25} {metrics_raw['CER']:>15.2f} {metrics_verified['CER']:>15.2f} {cer_imp:>+15.2f}")
    print(f"{'WER (%)':<25} {metrics_raw['WER']:>15.2f} {metrics_verified['WER']:>15.2f} {wer_imp:>+15.2f}")
    print(f"{'Word Accuracy (%)':<25} {metrics_raw['Word_Accuracy']:>15.2f} {metrics_verified['Word_Accuracy']:>15.2f} {acc_imp:>+15.2f}")
    print("=" * 70)
    
    # Show sample corrections
    print("\n📝 SAMPLE CORRECTIONS (First 5):")
    corrections_shown = 0
    for raw, verified, target in zip(predictions_raw, predictions_verified, targets):
        if raw != verified and corrections_shown < 5:
            print(f"\n  Raw:      '{raw}'")
            print(f"  Verified: '{verified}'")
            print(f"  Target:   '{target}'")
            if verified.strip().lower() == target.strip().lower():
                print(f"  ✅ FIXED!")
            corrections_shown += 1
    
    # Save results
    output_dir = TEST_CONFIG["output_dir"]
    save_results_with_comparison(
        predictions_raw, predictions_verified, targets, image_paths,
        metrics_raw, metrics_verified, ver_stats, output_dir
    )
    
    print(f"\n✅ Test complete! Full results in: {output_dir}")
    
    return metrics_raw, metrics_verified, predictions_verified, targets


# =====================================================================
# RUN TEST
# =====================================================================
if __name__ == "__main__":
    # Run test with agentic verification
    metrics_raw, metrics_verified, predictions, targets = test_model_with_verification()

TESTING GPT-2 BASED HTR MODEL WITH AGENTIC VERIFICATION

Loading tokenizer...
BOS=0 | EOS=2

Initializing Agentic Verification Module...
Agentic Verifier Lexicon: 6231 words | Max freq: 2488

Creating test dataloader...
Test samples loaded: 3830

Using device: cuda


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loading GPT-2 pretrained weights...
  Loaded: 147 weight tensors
  Missing (new layers): 98
GPT-2 weights loaded successfully.
  Vocab: 50257 pretrained + 8 new tokens
  Cross-attention layers: randomly initialized

Loading checkpoint: /home/mca24-26/handwritten text detection/iam_htr_gpt_3.pth
  - Epoch: 42
  - Best WER: 22.2396%

Running inference with agentic verification...


Running inference with verification: 100%|██████████████████| 120/120 [00:17<00:00,  6.86it/s]


📊 Agentic Verification Stats:
  - Total corrections made: 1/3830
  - Correction rate: 0.03%

Calculating metrics (without verification)...
Calculating metrics (WITH verification)...

TEST RESULTS - WITH VS WITHOUT VERIFICATION

Metric                      Without Verif      With Verif     Improvement
----------------------------------------------------------------------
CER (%)                              1.45            1.45           +0.00
WER (%)                              2.22            2.22           +0.00
Word Accuracy (%)                   97.78           97.78           +0.00

📝 SAMPLE CORRECTIONS (First 5):

  Raw:      'Diefen'
  Verified: 'Diefen-'
  Target:   'Douglas'

✅ Results saved to: ./test_results_gpt_greedy_with_verification
  - verification_comparison.json: Before/after metrics
  - predictions_comparison.txt: All predictions with comparison

✅ Test complete! Full results in: ./test_results_gpt_greedy_with_verification


## 4 without tps stn

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, GPT2LMHeadModel,
                          GPT2Config)
import timm
import Levenshtein
from tqdm import tqdm
from collections import defaultdict

# =====================================================================
# 1. CONFIGURATION
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/ocr/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/ocr/iam_gpt_wo_tpsstn.pth"

IMG_HEIGHT  = 96
IMG_WIDTH   = 384
MAX_SEQ_LEN = 25
BEAM_SIZE   = 5
D_MODEL     = 768
BATCH_SIZE  = 16

# =====================================================================
# 2. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=BATCH_SIZE):
    train_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.RandomApply([T.ColorJitter(
            brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(
        data_dir, words_txt_path, processor=processor
    )
    total      = len(full_dataset)
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(42)
    indices    = torch.randperm(total, generator=generator).tolist()

    train_samples = [full_dataset.samples[i]
                     for i in indices[:train_size]]
    val_samples   = [full_dataset.samples[i]
                     for i in indices[train_size:]]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor,
                     transform, max_target_length=MAX_SEQ_LEN):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB',
                                  (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_loader = DataLoader(
        SplitDataset(train_samples, processor, train_transform),
        batch_size=batch_size, shuffle=True,
        num_workers=4, drop_last=True
    )
    val_loader = DataLoader(
        SplitDataset(val_samples, processor, val_transform),
        batch_size=batch_size, shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE – WITHOUT TPS
# =====================================================================

# ── Swin Encoder ─────────────────────────────────────────────────────
class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))


# ── GPT-2 Decoder ────────────────────────────────────────────────────
class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        print("Loading GPT-2 pretrained weights...")

        PRETRAINED_VOCAB = 50257

        config = GPT2Config.from_pretrained("gpt2")
        config.add_cross_attention = True
        config.vocab_size = vocab_size

        self.decoder = GPT2LMHeadModel(config)

        pretrained = GPT2LMHeadModel.from_pretrained("gpt2")
        pretrained_dict = pretrained.state_dict()

        mismatch_keys = {
            "transformer.wte.weight",
            "lm_head.weight"
        }
        filtered_dict = {
            k: v for k, v in pretrained_dict.items()
            if k not in mismatch_keys
        }

        load_result = self.decoder.load_state_dict(
            filtered_dict, strict=False
        )
        print(f"  Loaded: {len(filtered_dict)} weight tensors")
        print(f"  Missing (new layers): {len(load_result.missing_keys)}")

        with torch.no_grad():
            self.decoder.transformer.wte.weight\
                .data[:PRETRAINED_VOCAB].copy_(
                    pretrained_dict["transformer.wte.weight"]
                )
            self.decoder.lm_head.weight\
                .data[:PRETRAINED_VOCAB].copy_(
                    pretrained_dict["lm_head.weight"]
                )

        del pretrained, pretrained_dict
        print(f"GPT-2 weights loaded successfully.")
        print(f"  Vocab: {PRETRAINED_VOCAB} pretrained "
              f"+ {vocab_size - PRETRAINED_VOCAB} new tokens")
        print(f"  Cross-attention layers: randomly initialized")

    def forward(self, input_ids,
                encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )


# ── Full Pipeline – TPS REMOVED ─────────────────────────────────────
class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        # TPS layer removed
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model//2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.gpt2_decoder = GPT2Decoder(
            vocab_size=vocab_size, d_model=d_model
        )

    def _extract_visual_memory(self, images):
        # Direct pass to Swin (no TPS rectification)
        swin_feat = self.swin_encoder(images)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),
            dec_input
        )
        labels = target_ids[:, 1:].clone()

        outputs = self.gpt2_decoder(
            input_ids=dec_input,
            encoder_hidden_states=memory,
            labels=None
        )
        logits = outputs.logits

        if criterion is not None:
            return criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100
        )

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2,
                 beam_size=BEAM_SIZE):
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)

        if beam_size == 1:
            generated = torch.full(
                (B, 1), bos_token_id,
                dtype=torch.long, device=device
            )
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_length - 1):
                out = self.gpt2_decoder(
                    input_ids=generated,
                    encoder_hidden_states=memory,
                    labels=None
                )
                next_tokens = out.logits[:, -1, :].argmax(
                    dim=-1, keepdim=True
                )
                next_tokens = next_tokens.masked_fill(
                    finished.unsqueeze(1), eos_token_id
                )
                generated = torch.cat([generated, next_tokens], dim=1)
                finished |= (next_tokens.squeeze(1) == eos_token_id)
                if finished.all():
                    break
            return generated[:, 1:]

        all_results = []
        for b in range(B):
            mem = memory[b:b+1]
            beams = [(0.0, [bos_token_id])]
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor(
                        [tokens], dtype=torch.long, device=device
                    )
                    out = self.gpt2_decoder(
                        input_ids=inp,
                        encoder_hidden_states=mem,
                        labels=None
                    )
                    log_prob = F.log_softmax(
                        out.logits[0, -1], dim=-1
                    )
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(),
                                       top_id.tolist()):
                        candidates.append(
                            (score + lp, tokens + [tid])
                        )

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break

                if not beams:
                    break

            all_beams = completed if completed else beams
            _, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(
                best_tokens[1:], dtype=torch.long, device=device
            )
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded = torch.full(
            (B, max_len), eos_token_id,
            dtype=torch.long, device=device
        )
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded


# =====================================================================
# 4. AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon  = {}
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = defaultdict(int)
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] += 1
        self.lexicon  = dict(freq)
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon: {len(self.lexicon)} words | "
              f"Max freq: {self.freq_max}")

    def verify_and_correct(self, text_output,
                           top_logprob=None,
                           logprob_threshold=-0.3):
        cleaned = text_output.strip().lower()

        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if text_output[0].isupper() and len(text_output) > 1:
            if text_output[1:].islower():
                if cleaned in self.lexicon:
                    return text_output

        if top_logprob is not None:
            avg_lp = sum(top_logprob) / max(len(top_logprob), 1)
            if avg_lp > logprob_threshold:
                return text_output

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue
            freq_score = freq / self.freq_max
            score      = freq_score - (dist * 0.3)
            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# 5. METRICS & EARLY STOPPING
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 6. LLRD OPTIMIZER (without TPS)
# =====================================================================
def build_llrd_optimizer(model, base_lr=5e-5,
                         decay_factor=0.75,
                         weight_decay=0.05):
    assigned = set()

    def collect(named_params, filter_fn):
        params = []
        for name, param in named_params:
            if id(param) not in assigned and filter_fn(name):
                params.append(param)
                assigned.add(id(param))
        return params

    def collect_all(named_params):
        params = []
        for name, param in named_params:
            if id(param) not in assigned:
                params.append(param)
                assigned.add(id(param))
        return params

    gpt2_crossattn = collect(
        model.gpt2_decoder.named_parameters(),
        lambda n: "crossattention" in n or "cross_attn" in n
    )
    gpt2_selfattn = collect_all(
        model.gpt2_decoder.named_parameters()
    )
    bilstm_params = collect_all(model.bilstm.named_parameters())
    swin_proj = collect(
        model.swin_encoder.named_parameters(),
        lambda n: n.startswith("proj.") or n.startswith("norm.")
    )
    swin_s4 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_3" in n
    )
    swin_s3 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_2" in n
    )
    swin_s2 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_1" in n
    )
    swin_s1 = collect_all(
        model.swin_encoder.swin.named_parameters()
    )

    # TPS parameters removed – no tps_params group

    lr = [
        base_lr,
        base_lr * decay_factor,
        base_lr * decay_factor**2,
        base_lr * decay_factor**3,
        base_lr * decay_factor**4,
        base_lr * decay_factor**5,
        base_lr * decay_factor**6,
        base_lr * decay_factor**7,
    ]

    groups_raw = [
        (gpt2_crossattn, lr[0], "gpt2_crossattn"),
        (gpt2_selfattn,  lr[1], "gpt2_selfattn"),
        (bilstm_params,  lr[2], "bilstm"),
        (swin_proj,      lr[3], "swin_proj"),
        (swin_s4,        lr[4], "swin_stage4"),
        (swin_s3,        lr[5], "swin_stage3"),
        (swin_s2,        lr[6], "swin_stage2"),
        (swin_s1,        lr[7], "swin_stage1"),
    ]

    param_groups = [
        {"params": params, "lr": lrate, "name": name}
        for params, lrate, name in groups_raw
        if len(params) > 0
    ]

    total_assigned = sum(len(g["params"]) for g in param_groups)
    total_model    = sum(1 for _ in model.parameters())
    if total_assigned != total_model:
        print(f"WARNING: {total_model - total_assigned} params unassigned")

    print("\nLLRD Groups (without TPS):")
    print(f"{'Name':<20} {'LR':>10} {'Params':>10}")
    print("-" * 44)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<20} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


# =====================================================================
# 7. TRAINING LOOP
# =====================================================================
def run_training_pipeline(epochs=60, batch_size=BATCH_SIZE,
                          base_lr=5e-5,
                          early_stopping_patience=8,
                          resume_from=None):

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | "
          f"EOS={tokenizer.eos_token_id} | "
          f"PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE,
        tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_p = sum(p.numel() for p in model.parameters())
    print(f"Total params: {total_p/1e6:.2f}M")
    print(f"Resolution  : {IMG_HEIGHT}x{IMG_WIDTH}")
    print(f"Beam size   : {BEAM_SIZE}")

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100, label_smoothing=0.1
    )

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = build_llrd_optimizer(
        model, base_lr=base_lr,
        decay_factor=0.75, weight_decay=0.05
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )

    agent_verifier = AgenticVerificationModule(
        words_txt_file=WORDS_TXT_FILE
    )
    early_stopper  = EarlyStopping(
        patience=early_stopping_patience, min_delta=0.05
    )
    best_val_wer   = float('inf')
    start_epoch    = 1
    swin_unfrozen  = False

    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        if start_epoch > 4:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            swin_unfrozen = True
        print(f"Resumed epoch {start_epoch} | "
              f"Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")

    for epoch in range(start_epoch, epochs + 1):

        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = build_llrd_optimizer(
                model, base_lr=base_lr,
                decay_factor=0.75, weight_decay=0.05
            )
            scheduler = torch.optim.lr_scheduler \
                .CosineAnnealingWarmRestarts(
                    optimizer, T_0=10, T_mult=2, eta_min=1e-7
                )
            swin_unfrozen = True
            print("Swin encoder unfrozen — LLRD optimizer rebuilt.")

        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels,
                         criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        lrs = {g["name"]: g["lr"]
               for g in optimizer.param_groups}
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | "
              f"LR gpt2_cross: {lrs.get('gpt2_crossattn', 0):.2e} "
              f"| LR swin4: {lrs.get('swin_stage4', 0):.2e}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(
                    val_loader,
                    desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens = model.generate(
                    images,
                    max_length   = MAX_SEQ_LEN,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id,
                    beam_size    = 1
                )
                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(
                            tokens[i], skip_special_tokens=True
                        )
                        print(f"  Target: '{text_labels[i]}' "
                              f"| Pred: '{raw.strip()}'")
                    print("---")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(
                        tokens[i], skip_special_tokens=True
                    )
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | "
              f"CER: {metrics['CER']:.2f}% | "
              f"WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer':             best_val_wer,
                'cer':                  metrics['CER'],
                'img_height':           IMG_HEIGHT,
                'img_width':            IMG_WIDTH,
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | WER: {best_val_wer:.2f}%")

        if early_stopper(current_wer):
            print(f"Early stopping at epoch {epoch}.")
            break

        print("=" * 70)


# =====================================================================
# 8. MAIN
# =====================================================================
if __name__ == "__main__":
    run_training_pipeline(
        epochs                  = 60,
        batch_size              = BATCH_SIZE,
        base_lr                 = 5e-5,
        early_stopping_patience = 8,
        resume_from             = CHECKPOINT_PATH
    )

Loading tokenizer...
BOS=0 | EOS=2 | PAD=1
Registered 38305 samples.


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loading GPT-2 pretrained weights...
  Loaded: 147 weight tensors
  Missing (new layers): 98
GPT-2 weights loaded successfully.
  Vocab: 50257 pretrained + 8 new tokens
  Cross-attention layers: randomly initialized
Total params: 247.03M
Resolution  : 96x384
Beam size   : 5
Swin frozen for first 3 epochs.

LLRD Groups (without TPS):
Name                         LR     Params
--------------------------------------------
gpt2_crossattn         5.00e-05     28.37M
gpt2_selfattn          3.75e-05    124.45M
bilstm                 2.81e-05      7.09M
swin_proj              2.11e-05      0.40M
swin_stage4            1.58e-05     27.30M
swin_stage3            1.19e-05     57.31M
swin_stage2            8.90e-06      1.71M
swin_stage1            6.67e-06      0.40M
Lexicon: 6231 words | Max freq: 2488

Training on: cuda


Epoch 1/60 [Train]: 100%|████████████████| 2154/2154 [07:31<00:00,  4.77it/s, loss=3.9920]


Epoch 1 | Loss: 5.0417 | LR gpt2_cross: 4.88e-05 | LR swin4: 1.54e-05


...

Epoch 39 | Loss: 1.5168 | LR gpt2_cross: 4.73e-05 | LR swin4: 1.50e-05


Epoch 39/60 [Val]:   1%|▎                                 | 2/240 [00:00<00:47,  5.05it/s]


--- Debug Epoch 39 ---
  Target: 'any' | Pred: 'any'
  Target: 'any' | Pred: 'any'
  Target: 'unopposed' | Pred: 'uprospective'
---


Epoch 39/60 [Val]: 100%|████████████████████████████████| 240/240 [00:36<00:00,  6.61it/s]

Epoch 39 | CER: 15.68% | WER: 23.78%
EarlyStopping: 8/8
Early stopping at epoch 39.


In [1]:
# =====================================================================
# TEST SCRIPT FOR IAM HTR – GPT-2 PIPELINE (NO TPS)
# Swin + BiLSTM + GPT-2 + LLRD + Agentic Verification
# =====================================================================

import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer, GPT2LMHeadModel, GPT2Config
import timm
import Levenshtein
from tqdm import tqdm
import numpy as np
from collections import defaultdict

# =====================================================================
# 1. CONFIGURATION (must match training)
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/ocr/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/ocr/iam_gpt_wo_tpsstn.pth"

IMG_HEIGHT  = 96
IMG_WIDTH   = 384
MAX_SEQ_LEN = 25
BEAM_SIZE   = 5
D_MODEL     = 768
BATCH_SIZE  = 32          # can be larger for inference
NUM_WORKERS = 4
SEED        = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# =====================================================================
# 2. DATASET & DATALOADER (same as training)
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)


def get_iam_test_loader(data_dir, words_txt_path, processor, batch_size=32):
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor)
    total = len(full_dataset)
    # Use the same 90:10 split as training – test on the validation portion
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(SEED)
    indices    = torch.randperm(total, generator=generator).tolist()
    val_indices = indices[train_size:]   # 10% for testing

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=MAX_SEQ_LEN):
            self.samples = samples
            self.processor = processor
            self.transform = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            # labels are not needed for inference, but we keep for consistency
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    val_samples = [full_dataset.samples[i] for i in val_indices]
    val_dataset = SplitDataset(val_samples, processor, val_transform)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=NUM_WORKERS)
    return val_loader


tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
test_loader = get_iam_test_loader(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, BATCH_SIZE)

# =====================================================================
# 3. MODEL ARCHITECTURE (same as training – NO TPS)
# =====================================================================
class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))


class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        config = GPT2Config.from_pretrained("gpt2")
        config.add_cross_attention = True
        config.vocab_size = vocab_size
        self.decoder = GPT2LMHeadModel(config)
        # We don't need to load pretrained weights here – they are in checkpoint

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        # TPS removed
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model//2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.gpt2_decoder = GPT2Decoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        swin_feat = self.swin_encoder(images)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids=None, criterion=None):
        memory = self._extract_visual_memory(images)
        # Not used during inference, but kept for consistency
        return memory

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2,
                 beam_size=BEAM_SIZE):
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)

        if beam_size == 1:
            generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_length - 1):
                out = self.gpt2_decoder(input_ids=generated, encoder_hidden_states=memory)
                next_tokens = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                next_tokens = next_tokens.masked_fill(finished.unsqueeze(1), eos_token_id)
                generated = torch.cat([generated, next_tokens], dim=1)
                finished |= (next_tokens.squeeze(1) == eos_token_id)
                if finished.all():
                    break
            return generated[:, 1:]   # strip BOS

        # Beam search
        all_results = []
        for b in range(B):
            mem = memory[b:b+1]
            beams = [(0.0, [bos_token_id])]
            completed = []
            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor([tokens], dtype=torch.long, device=device)
                    out = self.gpt2_decoder(input_ids=inp, encoder_hidden_states=mem)
                    log_prob = F.log_softmax(out.logits[0, -1], dim=-1)
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                        candidates.append((score + lp, tokens + [tid]))
                if not candidates:
                    break
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break
                if not beams:
                    break
            all_beams = completed if completed else beams
            _, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded = torch.full((B, max_len), eos_token_id, dtype=torch.long, device=device)
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded

# =====================================================================
# 4. AGENTIC VERIFICATION (same as training)
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon  = {}
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = defaultdict(int)
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] += 1
        self.lexicon  = dict(freq)
        self.freq_max = max(freq.values()) if freq else 1

    def verify_and_correct(self, text_output,
                           top_logprob=None,
                           logprob_threshold=-0.3):
        cleaned = text_output.strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if text_output[0].isupper() and len(text_output) > 1:
            if text_output[1:].islower() and cleaned in self.lexicon:
                return text_output

        if top_logprob is not None:
            avg_lp = sum(top_logprob) / max(len(top_logprob), 1)
            if avg_lp > logprob_threshold:
                return text_output

        best_candidate = None
        best_score = -float('inf')
        target_len = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue
            freq_score = freq / self.freq_max
            score = freq_score - (dist * 0.3)
            if score > best_score:
                best_score = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate

# =====================================================================
# 5. METRICS
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }

# =====================================================================
# 6. MAIN TEST
# =====================================================================
def test(use_beam_search=True, beam_size=BEAM_SIZE):
    print("="*60)
    print("TESTING IAM HTR – GPT-2 PIPELINE (NO TPS)")
    print("="*60)

    # Load model
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(DEVICE)
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"Checkpoint not found: {CHECKPOINT_PATH}")
        return

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')} (Best WER: {checkpoint.get('best_wer', '?')}%)")

    agent_verifier = AgenticVerificationModule(WORDS_TXT_FILE)

    model.eval()
    all_preds = []
    all_targets = []

    print(f"\nRunning inference with {'beam search' if use_beam_search else 'greedy'} "
          f"(beam_size={beam_size if use_beam_search else 1})...")

    with torch.no_grad():
        for images, _, text_labels in tqdm(test_loader, desc="Processing"):
            images = images.to(DEVICE)
            tokens = model.generate(
                images,
                max_length=MAX_SEQ_LEN,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                beam_size=beam_size
            )

            for i in range(images.size(0)):
                raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                verified = agent_verifier.verify_and_correct(raw)
                all_preds.append(verified)
                all_targets.append(text_labels[i])

    # Compute metrics
    metrics = calculate_metrics(all_preds, all_targets)
    print("\n" + "="*60)
    print("TEST RESULTS")
    print("="*60)
    print(f"Total samples: {len(all_targets)}")
    print(f"Character Error Rate (CER): {metrics['CER']:.2f}%")
    print(f"Word Error Rate (WER): {metrics['WER']:.2f}%")
    print("="*60)

    print("\nSample predictions (first 10):")
    for i in range(min(10, len(all_preds))):
        print(f"GT: {all_targets[i]}")
        print(f"PR: {all_preds[i]}")
        print("-"*40)

if __name__ == "__main__":
    # Run with beam search (default beam=5). Set use_beam_search=False for greedy.
    test(use_beam_search=True, beam_size=BEAM_SIZE)

Using device: cuda


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Registered 38305 samples.
TESTING IAM HTR – GPT-2 PIPELINE (NO TPS)


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loaded checkpoint from epoch 31 (Best WER: 22.6834%)

Running inference with beam search (beam_size=5)...


Processing: 100%|█████████████████████████████████████| 120/120 [2:02:38<00:00, 61.32s/it]


TEST RESULTS
Total samples: 3831
Character Error Rate (CER): 13.89%
Word Error Rate (WER): 22.06%

Sample predictions (first 10):
GT: any
PR: any
----------------------------------------
GT: any
PR: any
----------------------------------------
GT: unopposed
PR: manager
----------------------------------------
GT: conference
PR: conference
----------------------------------------
GT: Sir
PR: Sir
----------------------------------------
GT: -
PR: -
----------------------------------------
GT: the
PR: the
----------------------------------------
GT: United
PR: United
----------------------------------------
GT: Mr.
PR: Mr.
----------------------------------------
GT: drastic
PR: decision
----------------------------------------
